### 📈 A Quant's Guide to Modeling Tail-Risk and Black Swan Events

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Learn to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### 📈 Empirical Returns and Black Swan Events

Stock returns are typically treated as random variables, their distributions and subsequent statistical properties are subject for discussion as they change over time.

University professors will have you visualize returns as a distribution but leave out critical details as many lack the necessary quantitative foundations to understand the compression of information 

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# -----------------------------------------------------------------------------
# Load SPY data (1999-2026). Assumes @spy_1999_2026.csv is available.
# Uses Close-to-Close returns. Adjust column names as needed.
# -----------------------------------------------------------------------------

FILE = "spy_1999_2026.csv"

# Try to support common CSV column names ("Date","Close","Adj Close",etc.)
df = pd.read_csv(FILE)
date_column = [c for c in df.columns if c.strip().lower() in ["date"]][0]
close_column = None
for c in df.columns:
    if c.lower().replace(" ", "") in ["adjclose", "adj close"]:
        close_column = c
        break
if close_column is None:
    # fallback to "Close" if Adj Close doesn't exist
    close_column = [c for c in df.columns if c.strip().lower() == "close"][0]

df = df[[date_column, close_column]].copy()
df.rename(columns={date_column: "Date", close_column: "Close"}, inplace=True)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Calculate close-to-close daily return in decimal form
df["Return"] = df["Close"].pct_change()

# -----------------------------------------------------------------------------
START_DATE = "1999-01-01"
STD_THRESHOLD = 5.0
BIN_WIDTH_PERCENT = 0.25
MIN_BINS = 80
HISTOGRAM_Y_PADDING = 1.20
FRAME_DURATION_MS = 350
TRANSITION_DURATION_MS = 150

# --- Clean / sort source data -------------------------------------------------
df_sub = df.copy()
df_sub = (
    df_sub.loc[df_sub["Date"] >= START_DATE, ["Date", "Return"]]
    .dropna(subset=["Date", "Return"])
    .sort_values("Date")
    .reset_index(drop=True)
)

returns = df_sub["Return"].astype(float)

# --- GLOBAL (not rolling) mean and std for black swan threshold --------------
global_mean = returns.mean()
global_std = returns.std(ddof=1)

df_sub["global_mean"] = global_mean
df_sub["global_std"] = global_std
df_sub["upper_band"] = global_mean + STD_THRESHOLD * global_std
df_sub["lower_band"] = global_mean - STD_THRESHOLD * global_std

df_sub["z_score"] = (
    (returns - global_mean) / global_std
)

df_sub["black_swan"] = (
    df_sub["z_score"].abs().gt(STD_THRESHOLD)
    & df_sub["z_score"].notna()
    & (global_std > 0)
)

# This is the true cumulative counter across the full history.
df_sub["black_swan_count_cumulative"] = df_sub["black_swan"].cumsum()
df_sub["return_pct"] = returns * 100
for col in ["global_mean", "global_std", "upper_band", "lower_band"]:
    df_sub[f"{col}_pct"] = df_sub[col] * 100

df_sub["year"] = df_sub["Date"].dt.year

# One animation frame per year: use the final trading day index for each year.
years = df_sub["year"].drop_duplicates().to_list()
year_end_indices = (
    df_sub.groupby("year", sort=True)
    .tail(1)
    .index
    .to_numpy()
)
year_to_end_idx = dict(zip(years, year_end_indices))
year_to_black_swan_count = df_sub.groupby("year")["black_swan"].sum().astype(int).to_dict()

# --- Histogram binning --------------------------------------------------------
return_min = df_sub["return_pct"].min()
return_max = df_sub["return_pct"].max()
bin_min = np.floor(return_min / BIN_WIDTH_PERCENT) * BIN_WIDTH_PERCENT
bin_max = np.ceil(return_max / BIN_WIDTH_PERCENT) * BIN_WIDTH_PERCENT

if int((bin_max - bin_min) / BIN_WIDTH_PERCENT) < MIN_BINS:
    midpoint = (bin_min + bin_max) / 2
    half_range = (MIN_BINS * BIN_WIDTH_PERCENT) / 2
    bin_min = midpoint - half_range
    bin_max = midpoint + half_range

bin_edges = np.arange(bin_min, bin_max + BIN_WIDTH_PERCENT, BIN_WIDTH_PERCENT)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
n_bins = len(bin_centers)

bin_index = np.digitize(df_sub["return_pct"], bin_edges, right=False) - 1
bin_index = np.clip(bin_index, 0, n_bins - 1)
df_sub["bin_index"] = bin_index

# Cumulative histogram by day; yearly frames sample this at each year-end index.
incremental_counts = np.zeros((len(df_sub), n_bins), dtype=np.int32)
incremental_counts[np.arange(len(df_sub)), bin_index] = 1
cumulative_counts = np.cumsum(incremental_counts, axis=0)

full_counts = cumulative_counts[-1]
y_max = max(5, int(np.ceil(full_counts.max() * HISTOGRAM_Y_PADDING)))
x_min = bin_edges[0]
x_max = bin_edges[-1]

# --- Styling ------------------------------------------------------------------
off_white = "#e0e0e0"
histogram_color = "#00d4ff"
black_swan_color = "#f25b37"
mean_color = "#ffaa33"
threshold_color = "#aaaaaa"
baseline_color = "#777777"
counter_color = "#ffaa33"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# --- Helpers ------------------------------------------------------------------
def vertical_lines(xs, y_top):
    """Build Plotly x/y arrays for multiple vertical line segments."""
    x_vals, y_vals = [], []
    for x in xs:
        x_vals.extend([x, x, None])
        y_vals.extend([0, y_top, None])
    return x_vals, y_vals

def black_swan_lines_through_index(idx: int):
    """Show every detected black-swan event cumulatively through this frame."""
    mask = (df_sub.index.values <= idx) & df_sub["black_swan"].values
    event_returns = df_sub.loc[mask, "return_pct"].to_numpy()
    return vertical_lines(event_returns, y_max)

def current_cutoff_lines_for_index(idx: int):
    """
    For a yearly frame, display the global mean and ±5σ bands (same for all years).
    """
    # No longer variable by year/end, just use the global constant values
    mean = df_sub.iloc[0]["global_mean_pct"]
    upper = df_sub.iloc[0]["upper_band_pct"]
    lower = df_sub.iloc[0]["lower_band_pct"]
    mean_x, mean_y = [mean, mean], [0, y_max]
    upper_x, upper_y = [upper, upper], [0, y_max]
    lower_x, lower_y = [lower, lower], [0, y_max]
    return mean_x, mean_y, upper_x, upper_y, lower_x, lower_y

def counter_annotation_for_index(idx: int):
    current = df_sub.loc[idx]
    current_year = int(current["year"])
    cumulative_count = int(current["black_swan_count_cumulative"])
    year_count = int(year_to_black_swan_count.get(current_year, 0))
    pretty_date = current["Date"].strftime("%b %d, %Y")
    band_text = (
        f"Global Mean: {df_sub.iloc[0]['global_mean_pct']:.2f}% &nbsp; | &nbsp; "
        f"Bands: {df_sub.iloc[0]['lower_band_pct']:.2f}% to {df_sub.iloc[0]['upper_band_pct']:.2f}%"
    )
    return dict(
        x=0.985,
        y=0.965,
        xref="paper",
        yref="paper",
        xanchor="right",
        yanchor="top",
        align="right",
        text=(
            f"<b>Black Swan Events</b> <span style='font-size:16px'>&nbsp;|&nbsp;Through {current_year}</span><br>"
            f"<span style='font-size:24px;'>&nbsp;</span><br>"
            f"<span style='font-size:30px'>{cumulative_count}</span><br>"
            f"<span style='font-size:11px'>+{year_count} in {current_year} · Through {pretty_date}</span><br>"
            f"<span style='font-size:10px'>{band_text}</span>"
        ),
        showarrow=False,
        bgcolor="rgba(30,30,30,0.84)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        borderpad=7,
        font=dict(color=counter_color, size=13),
    )

# --- Initial state: first yearly frame ---------------------------------------
initial_year = years[0]
initial_idx = year_to_end_idx[initial_year]
initial_counts = cumulative_counts[initial_idx]
initial_outlier_x, initial_outlier_y = black_swan_lines_through_index(initial_idx)
(
    initial_mean_x,
    initial_mean_y,
    initial_upper_x,
    initial_upper_y,
    initial_lower_x,
    initial_lower_y,
) = current_cutoff_lines_for_index(initial_idx)

subtitle = (
    "<b>SPY Daily Returns & Black Swan 5σ Events</b><br>"
    f"Rolling calendar years, SPY from {START_DATE[:4]}.<br>"
    "A Black Swan event is |return − global mean| > 5 × global std."
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=bin_centers,
        y=initial_counts,
        width=BIN_WIDTH_PERCENT * 0.92,
        marker=dict(color=histogram_color),
        opacity=0.78,
        name="Daily Return Distribution",
        hovertemplate="Return bucket: %{x:.2f}%<br>Observations: %{y:,}<extra></extra>",
    )
)

fig.add_trace(
    go.Scatter(
        x=initial_outlier_x,
        y=initial_outlier_y,
        mode="lines",
        line=dict(color=black_swan_color, width=2),
        opacity=0.85,
        name="Cumulative Black Swan 5σ Event",
        hoverinfo="skip",
    )
)

fig.add_trace(
    go.Scatter(
        x=initial_mean_x,
        y=initial_mean_y,
        mode="lines",
        line=dict(color=mean_color, width=2, dash="dash"),
        name="Global Mean",
        hoverinfo="skip",
    )
)

fig.add_trace(
    go.Scatter(
        x=initial_upper_x,
        y=initial_upper_y,
        mode="lines",
        line=dict(color=threshold_color, width=1.5, dash="dot"),
        opacity=0.85,
        name="±5σ Global Cutoff",
        hoverinfo="skip",
    )
)

# Remove the duplicated "-5σ Cutoff, Year-End" from legend, keep only one ±5σ trace.
fig.add_trace(
    go.Scatter(
        x=initial_lower_x,
        y=initial_lower_y,
        mode="lines",
        line=dict(color=threshold_color, width=1.5, dash="dot"),
        opacity=0.85,
        name="",  # No legend entry
        hoverinfo="skip",
        showlegend=False,
    )
)

fig.add_vline(
    x=0,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.70,
)

# --- Frames: exactly one frame per year --------------------------------------
frames = []
slider_steps = []

for year in years:
    idx = year_to_end_idx[year]
    frame_name = f"year_{year}"
    counts = cumulative_counts[idx]
    outlier_x, outlier_y = black_swan_lines_through_index(idx)
    mean_x, mean_y, upper_x, upper_y, lower_x, lower_y = current_cutoff_lines_for_index(idx)

    frames.append(
        go.Frame(
            name=frame_name,
            data=[
                go.Bar(
                    x=bin_centers,
                    y=counts,
                    width=BIN_WIDTH_PERCENT * 0.92,
                    marker=dict(color=histogram_color),
                    opacity=0.78,
                ),
                go.Scatter(
                    x=outlier_x,
                    y=outlier_y,
                    mode="lines",
                    line=dict(color=black_swan_color, width=2),
                    opacity=0.85,
                ),
                go.Scatter(
                    x=mean_x,
                    y=mean_y,
                    mode="lines",
                    line=dict(color=mean_color, width=2, dash="dash"),
                ),
                go.Scatter(
                    x=upper_x,
                    y=upper_y,
                    mode="lines",
                    line=dict(color=threshold_color, width=1.5, dash="dot"),
                    opacity=0.85,
                ),
                go.Scatter(
                    x=lower_x,
                    y=lower_y,
                    mode="lines",
                    line=dict(color=threshold_color, width=1.5, dash="dot"),
                    opacity=0.85,
                    showlegend=False,
                ),
            ],
            traces=[0, 1, 2, 3, 4],
            layout=go.Layout(annotations=[counter_annotation_for_index(idx)]),
        )
    )

    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": FRAME_DURATION_MS, "redraw": True},
                    "transition": {"duration": TRANSITION_DURATION_MS, "easing": "cubic-in-out"},
                    "mode": "immediate",
                    "fromcurrent": True,
                },
            ],
            "label": str(year),
            "method": "animate",
        }
    )

fig.frames = frames

fig.update_layout(
    title=dict(text=subtitle, x=0.5, font=dict(color=off_white)),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=115, b=170, r=60, l=75),
    bargap=0.02,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.16,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
    ),
    hovermode="closest",
    annotations=[counter_annotation_for_index(initial_idx)],
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": FRAME_DURATION_MS, "redraw": True},
                            "transition": {"duration": TRANSITION_DURATION_MS, "easing": "cubic-in-out"},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "transition": {"duration": 0},
                            "mode": "immediate",
                            "fromcurrent": True,
                        },
                    ],
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {"size": 14, "color": off_white},
                "prefix": "Year: ",
                "visible": True,
                "xanchor": "right",
            },
            "transition": {"duration": 0},
            "pad": {"b": 40, "t": 30},
            "len": 0.85,
            "x": 0.15,
            "y": -0.08,
            "steps": slider_steps,
        }
    ],
)

fig.update_xaxes(
    axis_style,
    range=[x_min, x_max],
    title_text="Daily Return",
    ticksuffix="%",
    tickformat=",.2f",
)

fig.update_yaxes(
    axis_style,
    range=[0, y_max],
    title_text="Frequency",
    tickformat=",d",
)

fig.show()


---

##### 📊 Failure of Static Parametric Modeling

In naive quantitative fashion, let's model the probability of these so-called Black Swans

 $$
 f(x) = \frac{1}{\sqrt{2\pi}\,\sigma} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)
 $$

We will take a parametric approach and calibrate a normal distributions mean and standard deviation and treat these events as a geometric random variable

In [ ]:
"""
Rewritten Plotly animation: SPY empirical 5-sigma events vs. normal-model waiting time.

What this version fixes:
  - The left subplot is static for the entire animation. No frame ever touches it.
  - The right x-axis is truly log-scaled, including the x-axis range.
  - The right subplot shows discrete random normal draws, not a smoothed z-score curve.
  - The random draws are precomputed once, then revealed frame-by-frame, so playback does
    not jitter from re-sampling or re-layouting.
  - The text overlay is now placed at the true bottom-left corner of the right subplot, without any background/panel.
  - Animation uses redraw=False, fixed-length arrays, and full trace payloads so static traces cannot disappear.
Plotly modebar tools are restored for full interactivity (zoom, pan, region selection).
  - Overlay text is shifted up and to the right by ~100px via `textposition` and `textfont` arguments.
"""

from __future__ import annotations

import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# =============================================================================
# Settings
# =============================================================================
FILE = "spy_1999_2026.csv"
START_DATE = "1999-01-01"

STD_THRESHOLD = 5.0
BIN_WIDTH_PERCENT = 0.25
MIN_BINS = 80
HISTOGRAM_Y_PADDING = 1.20
TRADING_DAYS_PER_YEAR = 252

RANDOM_SEED = 7
USE_RANDOM_GEOMETRIC_DRAW = False

N_RANDOM_DRAWS_DISPLAYED = 430
N_SIMULATION_FRAMES = 115
FRAME_DURATION_MS = 65
TRANSITION_DURATION_MS = 0
RIGHT_X_PADDING = 1.08


# =============================================================================
# Style
# =============================================================================
OFF_WHITE = "#e0e0e0"
HISTOGRAM_COLOR = "#00d4ff"
NORMAL_COLOR = "#39FF14"
BLACK_SWAN_COLOR = "#f25b37"
MEAN_COLOR = "#ffaa33"
THRESHOLD_COLOR = "#aaaaaa"
BASELINE_COLOR = "#777777"
COUNTER_COLOR = "#ffaa33"
PATH_COLOR = "#00d4ff"
MAX_PATH_COLOR = "#ffaa33"

TEXT_LINE_COLORS = [
    "#ffaa33",  # line 1: title
    "#8f8f98",  # line 2: status (waiting... now gray)
    "#00d4ff",  # line 3: days simulated
    "#39ff14",  # line 4: years simulated
    "#aaaaaa",  # line 5: expected wait (use threshold color for contrast)
]

AXIS_STYLE = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=OFF_WHITE),
    linecolor=OFF_WHITE,
    zeroline=False,
    title_font=dict(color=OFF_WHITE),
    # fixedrange=True,  # DISABLED to restore modebar Zoom/Pan tools
)


# =============================================================================
# Data preparation
# =============================================================================
def load_return_data(file_path: str | Path) -> pd.DataFrame:
    try:
        existing_df = globals()["df"]
        if {"Date", "Return"}.issubset(existing_df.columns):
            out = existing_df[["Date", "Return"]].copy()
            out["Date"] = pd.to_datetime(out["Date"])
            return out
    except Exception:
        pass

    raw = pd.read_csv(file_path)
    date_column = next(c for c in raw.columns if c.strip().lower() == "date")

    close_column = None
    for c in raw.columns:
        normalized = c.lower().replace(" ", "")
        if normalized in {"adjclose", "adjustedclose"}:
            close_column = c
            break
    if close_column is None:
        close_column = next(c for c in raw.columns if c.strip().lower() == "close")

    out = raw[[date_column, close_column]].copy()
    out.columns = ["Date", "Close"]
    out["Date"] = pd.to_datetime(out["Date"])
    out = out.sort_values("Date").reset_index(drop=True)
    out["Return"] = out["Close"].pct_change()
    return out[["Date", "Return"]]


def prepare_sample(source_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Any]]:
    sample = source_df.copy()
    sample["Date"] = pd.to_datetime(sample["Date"])
    sample = (
        sample.loc[sample["Date"] >= START_DATE, ["Date", "Return"]]
        .dropna(subset=["Date", "Return"])
        .sort_values("Date")
        .reset_index(drop=True)
    )
    if len(sample) < 30:
        raise ValueError("Need at least 30 return observations after START_DATE.")

    returns = sample["Return"].astype(float)
    mu = float(returns.mean())
    sigma = float(returns.std(ddof=1))
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError("Return standard deviation must be positive and finite.")

    sample["return_pct"] = returns * 100.0
    sample["z_score"] = (returns - mu) / sigma
    sample["black_swan"] = sample["z_score"].abs().ge(STD_THRESHOLD)

    stats: dict[str, Any] = {
        "mu": mu,
        "sigma": sigma,
        "mu_pct": mu * 100.0,
        "sigma_pct": sigma * 100.0,
        "observations": int(len(sample)),
        "empirical_event_count": int(sample["black_swan"].sum()),
        "excess_kurtosis": float(returns.kurt()),
        "first_date": sample["Date"].min().strftime("%b %d, %Y"),
        "last_date": sample["Date"].max().strftime("%b %d, %Y"),
    }
    return sample, stats


def build_histogram_model(sample: pd.DataFrame, stats: dict[str, Any]) -> dict[str, Any]:
    returns_pct = sample["return_pct"].to_numpy()
    mu_pct = float(stats["mu_pct"])
    sigma_pct = float(stats["sigma_pct"])
    observations = int(stats["observations"])

    lower_cutoff_pct = mu_pct - STD_THRESHOLD * sigma_pct
    upper_cutoff_pct = mu_pct + STD_THRESHOLD * sigma_pct

    return_min = float(np.nanmin(returns_pct))
    return_max = float(np.nanmax(returns_pct))

    bin_min = math.floor(min(return_min, lower_cutoff_pct) / BIN_WIDTH_PERCENT) * BIN_WIDTH_PERCENT
    bin_max = math.ceil(max(return_max, upper_cutoff_pct) / BIN_WIDTH_PERCENT) * BIN_WIDTH_PERCENT

    if int((bin_max - bin_min) / BIN_WIDTH_PERCENT) < MIN_BINS:
        midpoint = (bin_min + bin_max) / 2.0
        half_range = (MIN_BINS * BIN_WIDTH_PERCENT) / 2.0
        bin_min = midpoint - half_range
        bin_max = midpoint + half_range

    bin_edges = np.arange(bin_min, bin_max + BIN_WIDTH_PERCENT, BIN_WIDTH_PERCENT)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0
    hist_counts, _ = np.histogram(returns_pct, bins=bin_edges)
    y_max_left = max(5, int(math.ceil(hist_counts.max() * HISTOGRAM_Y_PADDING)))

    x_curve = np.linspace(bin_edges[0], bin_edges[-1], 900)
    normal_pdf = (1.0 / (sigma_pct * math.sqrt(2.0 * math.pi))) * np.exp(
        -0.5 * ((x_curve - mu_pct) / sigma_pct) ** 2
    )
    normal_counts = normal_pdf * observations * BIN_WIDTH_PERCENT

    return {
        "bin_edges": bin_edges,
        "bin_centers": bin_centers,
        "hist_counts": hist_counts,
        "y_max_left": y_max_left,
        "x_curve": x_curve,
        "normal_counts": normal_counts,
        "lower_cutoff_pct": lower_cutoff_pct,
        "upper_cutoff_pct": upper_cutoff_pct,
        "left_tail_mask": x_curve <= lower_cutoff_pct,
        "right_tail_mask": x_curve >= upper_cutoff_pct,
    }


# =============================================================================
# Simulation path: discrete random draws, precomputed once
# =============================================================================
def log_spaced_unique_days(end_day: int, n_points: int) -> np.ndarray:
    days = np.unique(np.rint(np.geomspace(1, end_day, n_points)).astype(int))
    days = days[(days >= 1) & (days <= end_day)]
    if len(days) == 0 or days[0] != 1:
        days = np.insert(days, 0, 1)
    if days[-1] != end_day:
        days = np.append(days, end_day)
    return days


def make_random_draw_path(wait_days: int, seed: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    draw_days = log_spaced_unique_days(wait_days, N_RANDOM_DRAWS_DISPLAYED)
    z_draws = rng.normal(0.0, 1.0, len(draw_days))
    pre_final = np.arange(len(draw_days)) < len(draw_days) - 1
    bad = pre_final & (np.abs(z_draws) >= STD_THRESHOLD)
    while np.any(bad):
        z_draws[bad] = rng.normal(0.0, 1.0, int(bad.sum()))
        bad = pre_final & (np.abs(z_draws) >= STD_THRESHOLD)
    n_spikes = min(10, max(3, len(draw_days) // 50))
    spike_pool = np.arange(8, max(9, len(draw_days) - 1))
    if len(spike_pool) >= n_spikes:
        spike_idx = rng.choice(spike_pool, size=n_spikes, replace=False)
        spike_signs = rng.choice([-1.0, 1.0], size=n_spikes)
        z_draws[spike_idx] = spike_signs * rng.uniform(2.35, 4.35, size=n_spikes)
    final_sign = -1.0 if rng.random() < 0.5 else 1.0
    z_draws[-1] = final_sign * STD_THRESHOLD
    running_max_abs_z = np.maximum.accumulate(np.abs(z_draws))
    return draw_days, z_draws, running_max_abs_z


def make_frame_days(wait_days: int) -> np.ndarray:
    frame_days = np.unique(np.rint(np.geomspace(1, wait_days, N_SIMULATION_FRAMES)).astype(int))
    frame_days = frame_days[(frame_days >= 1) & (frame_days <= wait_days)]
    if len(frame_days) == 0 or frame_days[0] != 1:
        frame_days = np.insert(frame_days, 0, 1)
    if frame_days[-1] != wait_days:
        frame_days = np.append(frame_days, wait_days)
    return frame_days


def mask_y_until(x: np.ndarray, y: np.ndarray, visible_through_day: int) -> np.ndarray:
    out = y.astype(float).copy()
    out[x > visible_through_day] = np.nan
    return out


def status_text(days_elapsed: int, wait_days: int, p_tail: float, expected_wait_days: float) -> str:
    years_elapsed = days_elapsed / TRADING_DAYS_PER_YEAR
    status = (
        f"<span style='color:{BLACK_SWAN_COLOR}'><b>5σ EVENT HIT</b></span>"
        if days_elapsed >= wait_days else
        f"<span style='color:{TEXT_LINE_COLORS[1]}'>waiting...</span>"
    )
    return (
        f"<span style='color:{TEXT_LINE_COLORS[0]}'><b>Normal model waiting time</b></span><br>"
        f"{status}<br>"
        f"<span style='color:{TEXT_LINE_COLORS[2]}'>Days simulated: <b>{days_elapsed:,.0f}</b></span><br>"
        f"<span style='color:{TEXT_LINE_COLORS[3]}'>Trading years: {years_elapsed:,.0f}</span><br>"
        f"<span style='color:{TEXT_LINE_COLORS[4]}'>E[wait] = {expected_wait_days:,.0f} trading days</span>"
    )


# =============================================================================
# Figure helpers
# =============================================================================
def vertical_lines(xs: np.ndarray, y_top: float):
    x_vals: list[float | None] = []
    y_vals: list[float | None] = []
    for x in xs:
        x_vals.extend([float(x), float(x), None])
        y_vals.extend([0.0, float(y_top), None])
    return x_vals, y_vals


def log_tick_values(xmax: int):
    max_power = int(math.floor(math.log10(xmax)))
    vals = [10**p for p in range(max_power + 1) if 10**p <= xmax]
    if all(abs(math.log10(xmax) - math.log10(v)) > 0.11 for v in vals):
        vals.append(xmax)
    vals = sorted(set(vals))
    labels = [f"{v:,}" for v in vals]
    return vals, labels


def add_left_static_panel(fig: go.Figure, sample: pd.DataFrame, stats: dict[str, Any], hist: dict[str, Any]) -> None:
    y_max_left = int(hist["y_max_left"])
    empirical_events = sample.loc[sample["black_swan"]]
    empirical_marker_x, empirical_marker_y = vertical_lines(
        empirical_events["return_pct"].to_numpy(), y_max_left
    )

    fig.add_trace(
        go.Bar(
            x=hist["bin_centers"],
            y=hist["hist_counts"],
            width=BIN_WIDTH_PERCENT * 0.92,
            marker=dict(color=HISTOGRAM_COLOR),
            opacity=0.72,
            name="Empirical daily returns",
            hovertemplate="Return bucket: %{x:.2f}%<br>Observations: %{y:,}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=hist["x_curve"],
            y=hist["normal_counts"],
            mode="lines",
            line=dict(color=NORMAL_COLOR, width=2.5),
            name="Calibrated normal",
            hovertemplate="Return: %{x:.2f}%<br>Normal expected count: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=hist["x_curve"][hist["left_tail_mask"]],
            y=hist["normal_counts"][hist["left_tail_mask"]],
            mode="lines",
            fill="tozeroy",
            line=dict(color=BLACK_SWAN_COLOR, width=0),
            fillcolor="rgba(242,91,55,0.18)",
            name="Theoretical 5σ tails",
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=hist["x_curve"][hist["right_tail_mask"]],
            y=hist["normal_counts"][hist["right_tail_mask"]],
            mode="lines",
            fill="tozeroy",
            line=dict(color=BLACK_SWAN_COLOR, width=0),
            fillcolor="rgba(242,91,55,0.18)",
            name="",
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=empirical_marker_x,
            y=empirical_marker_y,
            mode="lines",
            line=dict(color=BLACK_SWAN_COLOR, width=2.5),
            opacity=0.92,
            name="Empirical |Z| ≥ 5 days",
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[float(stats["mu_pct"]), float(stats["mu_pct"])],
            y=[0, y_max_left],
            mode="lines",
            line=dict(color=MEAN_COLOR, width=2, dash="dash"),
            name="Full-sample mean",
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[float(hist["upper_cutoff_pct"]), float(hist["upper_cutoff_pct"])],
            y=[0, y_max_left],
            mode="lines",
            line=dict(color=THRESHOLD_COLOR, width=1.7, dash="dot"),
            name="±5σ cutoff",
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[float(hist["lower_cutoff_pct"]), float(hist["lower_cutoff_pct"])],
            y=[0, y_max_left],
            mode="lines",
            line=dict(color=THRESHOLD_COLOR, width=1.7, dash="dot"),
            name="",
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )


# Removed add_static_overlay_background (no longer called/defined below).

# =============================================================================
# Main figure
# =============================================================================
def build_figure() -> go.Figure:
    source_df = load_return_data(FILE)
    sample, stats = prepare_sample(source_df)
    hist = build_histogram_model(sample, stats)

    observations = int(stats["observations"])
    p_tail = math.erfc(STD_THRESHOLD / math.sqrt(2.0))
    expected_wait_days = 1.0 / p_tail

    wait_rng = np.random.default_rng(RANDOM_SEED)
    wait_days = int(wait_rng.geometric(p_tail)) if USE_RANDOM_GEOMETRIC_DRAW else int(math.ceil(expected_wait_days))
    wait_years = wait_days / TRADING_DAYS_PER_YEAR

    draw_days, z_draws, running_max_abs_z = make_random_draw_path(wait_days, RANDOM_SEED + 1000)
    frame_days = make_frame_days(wait_days)
    initial_day = int(frame_days[0])
    right_xmax = int(math.ceil(wait_days * RIGHT_X_PADDING))

    subtitle = (
        "<b>SPY Daily Returns: Empirical 5σ Events vs. Normal Waiting Time</b><br>"
        f"Full-sample calibration from {stats['first_date']} through {stats['last_date']}. "
        f"normal expected wait: {wait_days:,} trading days ≈ {wait_years:,.0f} years."
    )

    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.58, 0.42],
        horizontal_spacing=0.085,
        subplot_titles=(
            "Empirical distribution + calibrated normal model",
            "Random normal draws until the first 5σ day",
        ),
    )

    # Left subplot: add once and preserve it in every animation frame.
    left_trace_start = len(fig.data)
    add_left_static_panel(fig, sample, stats, hist)
    left_trace_indices = list(range(left_trace_start, len(fig.data)))

    # Right subplot: animated traces only.
    initial_z = mask_y_until(draw_days, z_draws, initial_day)
    initial_running_max = mask_y_until(draw_days, running_max_abs_z, initial_day)

    random_draw_trace_index = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=draw_days,
            y=initial_z,
            mode="lines+markers",
            line=dict(color=PATH_COLOR, width=1.35, shape="linear"),
            marker=dict(size=4.5, color=PATH_COLOR, opacity=0.86),
            opacity=0.86,
            name="Random normal draws",
            connectgaps=False,
            hovertemplate="Day: %{x:,}<br>Random draw Z: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=2,
    )
    running_max_trace_index = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=draw_days,
            y=initial_running_max,
            mode="lines",
            line=dict(color=MAX_PATH_COLOR, width=3.0, shape="linear"),
            name="Running max |Z|",
            connectgaps=False,
            hovertemplate="Day: %{x:,}<br>Running max |Z|: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=2,
    )
    event_trace_index = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            mode="markers",
            marker=dict(size=17, color=BLACK_SWAN_COLOR, symbol="x", line=dict(width=2)),
            name="First normal 5σ event",
            hovertemplate="First event day: %{x:,}<br>Z: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=2,
    )

    # Text overlay trace. Now shifted up and right by ~100px using textposition='top right' & custom x/y.
    overlay_text_trace_index = len(fig.data)

    # To shift the overlay up and right by ~100px, we increase the x and y.
    # Since the x-axis is log-scale, increasing x more than 1 will push right.
    # y=-5.5 is close to the bottom; let's move y up by 1.5 units.
    # For x, moving from x=1 to x=2.5+ covers a good chunk on log scale.
    overlay_x = [1.5]  # empirically, 1 → 2.5 is significant shift on log-x
    overlay_y = [-4.75] # move up from -5.5 to -4.0

    fig.add_trace(
        go.Scatter(
            x=overlay_x,
            y=overlay_y,
            mode="text",
            text=[status_text(initial_day, wait_days, p_tail, expected_wait_days)],
            textposition="top right",
            textfont=dict(color=COUNTER_COLOR, size=13, family="Arial, sans-serif"),
            texttemplate="%{text}",
            cliponaxis=False,
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    # Static right-panel guide lines. Never animated.
    for y_value, name, showlegend in [
        (STD_THRESHOLD, "Simulation ±5σ barrier", True),
        (-STD_THRESHOLD, "", False),
        (0.0, "", False),
    ]:
        fig.add_trace(
            go.Scatter(
                x=[1, right_xmax],
                y=[y_value, y_value],
                mode="lines",
                line=dict(
                    color=THRESHOLD_COLOR if abs(y_value) == STD_THRESHOLD else BASELINE_COLOR,
                    width=1.5 if abs(y_value) == STD_THRESHOLD else 1.0,
                    dash="dot" if abs(y_value) == STD_THRESHOLD else "dash",
                ),
                name=name,
                showlegend=showlegend,
                hoverinfo="skip",
            ),
            row=1,
            col=2,
        )

    fig.add_vline(
        x=0,
        line=dict(color=BASELINE_COLOR, width=1, dash="dash"),
        opacity=0.65,
        row=1,
        col=1,
    )
    # Removed call to add_static_overlay_background(fig)

    for ann in fig.layout.annotations:
        ann.font = dict(color=OFF_WHITE, size=14)

    animated_trace_indices = [
        random_draw_trace_index,
        running_max_trace_index,
        event_trace_index,
        overlay_text_trace_index,
    ]
    all_trace_indices = list(range(len(fig.data)))

    frames: list[go.Frame] = []
    slider_steps: list[dict[str, Any]] = []
    final_z = float(z_draws[-1])

    def frame_trace_payload(frame_day: int, hit: bool) -> list[go.BaseTraceType]:
        """Return a full trace payload so static panels cannot disappear.

        The left subplot and static guide traces are included unchanged. The four
        right-side animated traces are cloned from their base traces and updated.
        This is slightly more verbose than partial frames, but it is much more
        reliable across notebook, browser, and exported HTML renderers.
        """
        payload: list[go.BaseTraceType] = []

        random_draw_trace = go.Scatter(fig.data[random_draw_trace_index])
        random_draw_trace.y = mask_y_until(draw_days, z_draws, frame_day)

        running_max_trace = go.Scatter(fig.data[running_max_trace_index])
        running_max_trace.y = mask_y_until(draw_days, running_max_abs_z, frame_day)

        event_trace = go.Scatter(fig.data[event_trace_index])
        event_trace.x = [wait_days] if hit else []
        event_trace.y = [final_z] if hit else []

        overlay_text_trace = go.Scatter(fig.data[overlay_text_trace_index])
        overlay_text_trace.text = [status_text(frame_day, wait_days, p_tail, expected_wait_days)]
        overlay_text_trace.x = overlay_x
        overlay_text_trace.y = overlay_y
        overlay_text_trace.textposition = "top right"

        replacements = {
            random_draw_trace_index: random_draw_trace,
            running_max_trace_index: running_max_trace,
            event_trace_index: event_trace,
            overlay_text_trace_index: overlay_text_trace,
        }

        for trace_index in all_trace_indices:
            payload.append(replacements.get(trace_index, fig.data[trace_index]))
        return payload

    for i, frame_day in enumerate(frame_days):
        frame_day = int(frame_day)
        frame_name = f"sim_{i:03d}"
        hit = frame_day >= wait_days

        frames.append(
            go.Frame(
                name=frame_name,
                data=frame_trace_payload(frame_day, hit),
                traces=all_trace_indices,
            )
        )

        label = f"{frame_day:,}" if i in {0, len(frame_days) - 1} or i % max(1, len(frame_days) // 8) == 0 else ""
        slider_steps.append(
            {
                "args": [
                    [frame_name],
                    {
                        "frame": {"duration": FRAME_DURATION_MS, "redraw": False},
                        "transition": {"duration": TRANSITION_DURATION_MS},
                        "mode": "immediate",
                    },
                ],
                "label": label,
                "method": "animate",
            }
        )

    fig.frames = frames

    right_tickvals, right_ticktext = log_tick_values(right_xmax)
    right_log_range = [0.0, math.log10(right_xmax)]  # Plotly expects log10 values for log-axis range.

    # Restoring modebar tools (zoom, select, pan, etc.) by updating config in fig.show()
    # If displaying in Jupyter, pass config to fig.show(config=...)
    # Show all standard modebar tools by passing 'displayModeBar': True and not hiding tools.
    plotly_config = {
        "displayModeBar": True,
        "modeBarButtonsToAdd": [],
        "modeBarButtonsToRemove": [],
        "scrollZoom": True,
        "displaylogo": False,
        # Additional config options can be toggled here
    }

    fig.update_layout(
        title=dict(text=subtitle, x=0.5, font=dict(color=OFF_WHITE)),
        template="plotly_dark",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        height=720,
        width=1200,
        margin=dict(t=125, b=175, r=70, l=70),
        bargap=0.02,
        hovermode="closest",
        showlegend=False,
        uirevision="left-panel-frozen-right-random-draws",
        updatemenus=[
            {
                "type": "buttons",
                "direction": "left",
                "showactive": False,
                "x": 0.105,
                "xanchor": "right",
                "y": 0,
                "yanchor": "top",
                "pad": {"r": 10, "t": 88},
                "buttons": [
                    {
                        "label": "▶ Play",
                        "method": "animate",
                        "args": [
                            None,
                            {
                                "frame": {"duration": FRAME_DURATION_MS, "redraw": False},
                                "transition": {"duration": TRANSITION_DURATION_MS},
                                "fromcurrent": True,
                                "mode": "immediate",
                            },
                        ],
                    },
                    {
                        "label": "⏸ Pause",
                        "method": "animate",
                        "args": [
                            [None],
                            {
                                "frame": {"duration": 0, "redraw": False},
                                "transition": {"duration": 0},
                                "mode": "immediate",
                            },
                        ],
                    },
                ],
            }
        ],
        sliders=[
            {
                "active": 0,
                "yanchor": "top",
                "xanchor": "left",
                "currentvalue": {
                    "font": {"size": 14, "color": OFF_WHITE},
                    "prefix": "",
                    "visible": True,
                    "xanchor": "right",
                },
                "transition": {"duration": 0},
                "pad": {"b": 40, "t": 30},
                "len": 0.84,
                "x": 0.15,
                "y": -0.08,
                "steps": slider_steps,
            }
        ],
    )

    fig.update_xaxes(
        AXIS_STYLE,
        range=[float(hist["bin_edges"][0]), float(hist["bin_edges"][-1])],
        title_text="Daily Return",
        ticksuffix="%",
        tickformat=",.2f",
        row=1,
        col=1,
    )
    fig.update_yaxes(
        AXIS_STYLE,
        range=[0, int(hist["y_max_left"])],
        title_text="Frequency",
        tickformat=",d",
        row=1,
        col=1,
    )
    fig.update_xaxes(
        AXIS_STYLE,
        type="log",
        range=right_log_range,
        tickvals=right_tickvals,
        ticktext=right_ticktext,
        title_text="Simulated Trading Days Waiting",
        row=1,
        col=2,
    )
    fig.update_yaxes(
        AXIS_STYLE,
        range=[-5.7, 5.7],
        title_text="Random Normal Draw, Z-Score",
        tickformat=",.1f",
        row=1,
        col=2,
    )

    return fig, plotly_config


fig, plotly_config = build_figure()
fig.show(config=plotly_config)


: 

Clearly, we severely underestimate tail risk for one of a variety of reasons: lack of empirical independence, that is return autocorrelation, excess kurtosis, so on and so forth...

You might be saying "Roman, that's a trivial approach - use neural networks or AI".

That is not a cure all for model and parameter risk as we are going to see...

---

 ##### 🌊 Modeling Conditional Distributions

In reality, a time series or return distribution is a compression of information.  In other words, impacts a stock's price?
* News, social sentiment
* Geopolitics, macroeconomy,
* Regulation, administrative policy

I don't know about you but I don't see that anywhere in a time series of a stock price or a return distribution.

These dimensions dramatically change the probability of *extreme* or *normal* returns, in practice we can capture this idea by using regime modeling

 $$
 \text{GARCH}(1,1):\quad \sigma_t^2 = \omega + \alpha \, \varepsilon_{t-1}^2 + \beta\, \sigma_{t-1}^2
 $$

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm

try:
    from arch import arch_model
except ImportError as exc:
    raise ImportError(
        "This snippet requires the `arch` package. Install it with: pip install arch"
    ) from exc


# -----------------------------------------------------------------------------
# Load SPY data. Assumes spy_1999_2026.csv is available.
# Uses close-to-close returns. Adjust FILE as needed.
# -----------------------------------------------------------------------------

FILE = "spy_1999_2026.csv"

df = pd.read_csv(FILE)

date_column = [c for c in df.columns if c.strip().lower() == "date"][0]

close_column = None
for c in df.columns:
    normalized = c.lower().replace(" ", "")
    if normalized in ["adjclose", "adjustedclose"]:
        close_column = c
        break

if close_column is None:
    close_column = [c for c in df.columns if c.strip().lower() == "close"][0]

df = df[[date_column, close_column]].copy()
df.rename(columns={date_column: "Date", close_column: "Close"}, inplace=True)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df["Return"] = df["Close"].pct_change()


# -----------------------------------------------------------------------------
# Parameters
# -----------------------------------------------------------------------------

START_DATE = "1999-01-01"

TAIL_STD_THRESHOLD = 5.0
BIN_WIDTH_PERCENT = 0.25
MIN_BINS = 80

HISTOGRAM_Y_PADDING = 1.20
CURVE_Y_PADDING = 1.25
VOL_Y_PADDING = 1.18

FRAME_DURATION_MS = 350
TRANSITION_DURATION_MS = 150

GARCH_P = 1
GARCH_Q = 1

REGIME_ORDER = ["Low Vol", "Med Vol", "High Vol"]

REALIZED_VOL_WINDOW = 21
ANNUALIZATION_FACTOR = np.sqrt(252)

VOL_FORECAST_HORIZON_DAYS = 63
VOL_FORECAST_BAND_LOOKBACK = 252
VOL_FORECAST_X_BUFFER_DAYS = 5

TRADING_DAYS_PER_YEAR = 252


# -----------------------------------------------------------------------------
# Clean data
# -----------------------------------------------------------------------------

df_sub = (
    df.loc[df["Date"] >= START_DATE, ["Date", "Return"]]
    .dropna(subset=["Date", "Return"])
    .sort_values("Date")
    .reset_index(drop=True)
)

df_sub["return_pct"] = df_sub["Return"].astype(float) * 100
df_sub["year"] = df_sub["Date"].dt.year


# -----------------------------------------------------------------------------
# Pooled tail definition
# Fixed pooled ±5σ cutoffs are used across both top panels.
# -----------------------------------------------------------------------------

pooled_full_mean = df_sub["return_pct"].mean()
pooled_full_std = df_sub["return_pct"].std(ddof=1)

lower_tail_cutoff = pooled_full_mean - TAIL_STD_THRESHOLD * pooled_full_std
upper_tail_cutoff = pooled_full_mean + TAIL_STD_THRESHOLD * pooled_full_std

df_sub["tail_event"] = (
    (df_sub["return_pct"] <= lower_tail_cutoff)
    | (df_sub["return_pct"] >= upper_tail_cutoff)
)


# -----------------------------------------------------------------------------
# GARCH volatility regimes
# -----------------------------------------------------------------------------

garch_returns = df_sub["return_pct"].astype(float)

garch_model = arch_model(
    garch_returns,
    mean="Constant",
    vol="GARCH",
    p=GARCH_P,
    q=GARCH_Q,
    dist="normal",
    rescale=False,
)

garch_result = garch_model.fit(disp="off")

df_sub["garch_vol_daily_pct"] = np.asarray(garch_result.conditional_volatility)
df_sub["garch_vol_ann_pct"] = df_sub["garch_vol_daily_pct"] * ANNUALIZATION_FACTOR

garch_params = garch_result.params

garch_mu = float(garch_params.get("mu", garch_params.get("Const", 0.0)))
garch_omega = float(garch_params["omega"])
garch_alpha = float(garch_params[[p for p in garch_params.index if p.startswith("alpha")][0]])
garch_beta = float(garch_params[[p for p in garch_params.index if p.startswith("beta")][0]])

df_sub["garch_resid_pct"] = df_sub["return_pct"] - garch_mu
df_sub["garch_var_daily_pct2"] = df_sub["garch_vol_daily_pct"] ** 2

low_cut, high_cut = df_sub["garch_vol_daily_pct"].quantile([1 / 3, 2 / 3])

df_sub["vol_regime"] = np.select(
    [
        df_sub["garch_vol_daily_pct"] <= low_cut,
        df_sub["garch_vol_daily_pct"] <= high_cut,
    ],
    [
        "Low Vol",
        "Med Vol",
    ],
    default="High Vol",
)

df_sub["vol_regime"] = pd.Categorical(
    df_sub["vol_regime"],
    categories=REGIME_ORDER,
    ordered=True,
)


# -----------------------------------------------------------------------------
# Realized volatility
# Bottom panel uses annualized 21-trading-day realized volatility.
# -----------------------------------------------------------------------------

df_sub["realized_vol_ann_pct"] = (
    df_sub["return_pct"]
    .rolling(REALIZED_VOL_WINDOW)
    .std(ddof=1)
    * ANNUALIZATION_FACTOR
)

vol_model_error = df_sub["realized_vol_ann_pct"] - df_sub["garch_vol_ann_pct"]


# -----------------------------------------------------------------------------
# Animation frame dates
# One frame per year, using the final trading day of each year.
# -----------------------------------------------------------------------------

years = df_sub["year"].drop_duplicates().to_list()

year_end_indices = (
    df_sub.groupby("year", sort=True)
    .tail(1)
    .index
    .to_numpy()
)

year_to_end_idx = dict(zip(years, year_end_indices))


# -----------------------------------------------------------------------------
# Histogram binning
# -----------------------------------------------------------------------------

return_min = min(df_sub["return_pct"].min(), lower_tail_cutoff)
return_max = max(df_sub["return_pct"].max(), upper_tail_cutoff)

bin_min = np.floor(return_min / BIN_WIDTH_PERCENT) * BIN_WIDTH_PERCENT
bin_max = np.ceil(return_max / BIN_WIDTH_PERCENT) * BIN_WIDTH_PERCENT

if int((bin_max - bin_min) / BIN_WIDTH_PERCENT) < MIN_BINS:
    midpoint = (bin_min + bin_max) / 2
    half_range = (MIN_BINS * BIN_WIDTH_PERCENT) / 2
    bin_min = midpoint - half_range
    bin_max = midpoint + half_range

bin_edges = np.arange(bin_min, bin_max + BIN_WIDTH_PERCENT, BIN_WIDTH_PERCENT)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
n_bins = len(bin_centers)

bin_index = np.digitize(df_sub["return_pct"], bin_edges, right=False) - 1
bin_index = np.clip(bin_index, 0, n_bins - 1)
df_sub["bin_index"] = bin_index

incremental_counts = np.zeros((len(df_sub), n_bins), dtype=np.int32)
incremental_counts[np.arange(len(df_sub)), bin_index] = 1
cumulative_counts = np.cumsum(incremental_counts, axis=0)

x_min = bin_edges[0]
x_max = bin_edges[-1]
x_grid = np.linspace(x_min, x_max, 1000)


# -----------------------------------------------------------------------------
# Styling
# -----------------------------------------------------------------------------

off_white = "#e0e0e0"

histogram_color = "#00d4ff"

pooled_color = "#ffaa33"
pooled_tail_fill = "rgba(242,91,55,0.30)"

cutoff_color = "#aaaaaa"
baseline_color = "#777777"
black_swan_color = "#ff3b30"

low_text_color = "#5fd0ff"
high_text_color = "#f25b37"

forecast_band_opacity = {
    "Low Vol": "rgba(95,208,255,0.16)",
    "Med Vol": "rgba(255,170,51,0.16)",
    "High Vol": "rgba(242,91,55,0.18)",
}

regime_colors = {
    "Low Vol": "#5fd0ff",
    "Med Vol": "#ffaa33",
    "High Vol": "#f25b37",
}

regime_fills = {
    "Low Vol": "rgba(95,208,255,0.18)",
    "Med Vol": "rgba(255,170,51,0.18)",
    "High Vol": "rgba(242,91,55,0.20)",
}

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)


# -----------------------------------------------------------------------------
# Helpers: distribution panels
# -----------------------------------------------------------------------------

def safe_mean_std(values: pd.Series):
    values = values.dropna().astype(float)
    n = len(values)

    if n < 2:
        return np.nan, np.nan, n

    mu = values.mean()
    sigma = values.std(ddof=1)

    if not np.isfinite(sigma) or sigma <= 0:
        return mu, np.nan, n

    return mu, sigma, n


def tail_probability(mu, sigma, lower_cutoff, upper_cutoff):
    if not np.isfinite(mu) or not np.isfinite(sigma) or sigma <= 0:
        return np.nan, np.nan, np.nan

    left_prob = norm.cdf(lower_cutoff, loc=mu, scale=sigma)
    right_prob = norm.sf(upper_cutoff, loc=mu, scale=sigma)
    total_prob = left_prob + right_prob

    return left_prob, right_prob, total_prob


def pooled_tail_probability(mu, sigma):
    return tail_probability(
        mu,
        sigma,
        lower_tail_cutoff,
        upper_tail_cutoff,
    )


def expected_bin_counts_curve(mu, sigma, n_obs):
    if (
        n_obs < 2
        or not np.isfinite(mu)
        or not np.isfinite(sigma)
        or sigma <= 0
    ):
        return np.zeros_like(x_grid)

    return norm.pdf(x_grid, loc=mu, scale=sigma) * n_obs * BIN_WIDTH_PERCENT


def filled_tail_area(y_curve, side):
    if side == "left":
        mask = x_grid <= lower_tail_cutoff
    elif side == "right":
        mask = x_grid >= upper_tail_cutoff
    else:
        raise ValueError("side must be 'left' or 'right'")

    x_tail = x_grid[mask]
    y_tail = y_curve[mask]

    if len(x_tail) < 2:
        return np.array([]), np.array([])

    x_area = np.concatenate([x_tail, x_tail[::-1]])
    y_area = np.concatenate([y_tail, np.zeros_like(y_tail)])

    return x_area, y_area


def vertical_line(x, y_top):
    return [x, x], [0, y_top]


def black_swan_lines_through_index(idx: int, y_top: float):
    mask = (df_sub.index.values <= idx) & df_sub["tail_event"].values
    event_returns = df_sub.loc[mask, "return_pct"].to_numpy()
    event_dates = df_sub.loc[mask, "Date"].dt.strftime("%b %d, %Y").to_numpy()

    x_vals = []
    y_vals = []
    customdata = []

    for event_return, event_date in zip(event_returns, event_dates):
        x_vals.extend([event_return, event_return, None])
        y_vals.extend([0, y_top, None])
        customdata.extend(
            [
                [event_date, event_return],
                [event_date, event_return],
                [None, None],
            ]
        )

    return x_vals, y_vals, customdata


def empirical_tail_stats(data: pd.DataFrame):
    n = len(data)
    count = int(data["tail_event"].sum())

    if n == 0:
        return count, np.nan

    return count, count / n


def line_customdata(prob, empirical_rate, n_obs, tail_count):
    return np.column_stack(
        [
            np.full(len(x_grid), prob),
            np.full(len(x_grid), empirical_rate),
            np.full(len(x_grid), n_obs),
            np.full(len(x_grid), tail_count),
        ]
    )


def tail_customdata(x_area, prob, empirical_rate, n_obs, tail_count):
    return np.column_stack(
        [
            np.full(len(x_area), prob),
            np.full(len(x_area), empirical_rate),
            np.full(len(x_area), n_obs),
            np.full(len(x_area), tail_count),
        ]
    )


def expected_wait_days(probability):
    if not np.isfinite(probability) or probability <= 0:
        return np.nan

    return 1.0 / probability


def format_wait_time(probability):
    wait_days = expected_wait_days(probability)

    if not np.isfinite(wait_days):
        return "n/a"

    wait_years = wait_days / TRADING_DAYS_PER_YEAR

    if wait_years >= 1000:
        return f"{wait_years:,.0f} yrs"

    if wait_years >= 10:
        return f"{wait_years:,.1f} yrs"

    if wait_days >= 252:
        return f"{wait_years:,.2f} yrs"

    return f"{wait_days:,.1f} days"


def regime_wait_stats_from_state(state):
    low_state = state["regimes"]["Low Vol"]
    high_state = state["regimes"]["High Vol"]

    low_mu = low_state["mu"]
    low_sigma = low_state["sigma"]
    high_mu = high_state["mu"]
    high_sigma = high_state["sigma"]

    if (
        not np.isfinite(low_mu)
        or not np.isfinite(low_sigma)
        or low_sigma <= 0
        or not np.isfinite(high_mu)
        or not np.isfinite(high_sigma)
        or high_sigma <= 0
    ):
        return {
            "low_prob": np.nan,
            "high_prob": np.nan,
        }

    low_reference_lower = low_mu - TAIL_STD_THRESHOLD * low_sigma
    low_reference_upper = low_mu + TAIL_STD_THRESHOLD * low_sigma

    _, _, low_prob = tail_probability(
        low_mu,
        low_sigma,
        low_reference_lower,
        low_reference_upper,
    )

    _, _, high_prob = tail_probability(
        high_mu,
        high_sigma,
        low_reference_lower,
        low_reference_upper,
    )

    return {
        "low_prob": low_prob,
        "high_prob": high_prob,
    }


def wait_time_annotation_for_state(state):
    wait_stats = regime_wait_stats_from_state(state)

    low_prob = wait_stats["low_prob"]
    high_prob = wait_stats["high_prob"]

    if not np.isfinite(low_prob) or not np.isfinite(high_prob):
        text = (
            "<b>Black Swan Wait Time</b><br>"
            f"<span style='color:{low_text_color}'><b>Low Vol:</b> n/a</span><br>"
            f"<span style='color:{high_text_color}'><b>High Vol:</b> n/a</span>"
        )
    else:
        text = (
            "<b>Black Swan Wait Time</b><br>"
            f"<span style='color:{low_text_color}'><b>Low Vol:</b> {format_wait_time(low_prob)}</span><br>"
            f"<span style='color:{high_text_color}'><b>High Vol:</b> {format_wait_time(high_prob)}</span>"
        )

    return dict(
        x=0.985,
        y=0.965,
        xref="x3 domain",
        yref="y3 domain",
        xanchor="right",
        yanchor="top",
        align="right",
        text=text,
        showarrow=False,
        bgcolor="rgba(10,10,10,0.72)",
        bordercolor="rgba(255,255,255,0.22)",
        borderwidth=1,
        borderpad=7,
        font=dict(color=off_white, size=13),
    )


def build_distribution_state(idx: int):
    data = df_sub.iloc[: idx + 1].copy()

    histogram_counts = cumulative_counts[idx]

    pooled_mu, pooled_sigma, pooled_n = safe_mean_std(data["return_pct"])
    pooled_curve = expected_bin_counts_curve(pooled_mu, pooled_sigma, pooled_n)

    pooled_left_prob, pooled_right_prob, pooled_total_prob = pooled_tail_probability(
        pooled_mu,
        pooled_sigma,
    )

    pooled_tail_count, pooled_empirical_tail_rate = empirical_tail_stats(data)

    regime_states = {}

    for regime in REGIME_ORDER:
        regime_data = data.loc[data["vol_regime"] == regime]

        regime_mu, regime_sigma, regime_n = safe_mean_std(regime_data["return_pct"])
        regime_curve = expected_bin_counts_curve(regime_mu, regime_sigma, regime_n)

        left_prob, right_prob, total_prob = pooled_tail_probability(
            regime_mu,
            regime_sigma,
        )

        tail_count, empirical_tail_rate = empirical_tail_stats(regime_data)

        regime_states[regime] = {
            "mu": regime_mu,
            "sigma": regime_sigma,
            "n": regime_n,
            "curve": regime_curve,
            "left_prob": left_prob,
            "right_prob": right_prob,
            "total_prob": total_prob,
            "tail_count": tail_count,
            "empirical_tail_rate": empirical_tail_rate,
        }

    return {
        "histogram_counts": histogram_counts,
        "pooled_mu": pooled_mu,
        "pooled_sigma": pooled_sigma,
        "pooled_n": pooled_n,
        "pooled_curve": pooled_curve,
        "pooled_left_prob": pooled_left_prob,
        "pooled_right_prob": pooled_right_prob,
        "pooled_total_prob": pooled_total_prob,
        "pooled_tail_count": pooled_tail_count,
        "pooled_empirical_tail_rate": pooled_empirical_tail_rate,
        "regimes": regime_states,
    }


def make_black_swan_trace(idx: int, y_top: float):
    swan_x, swan_y, swan_cd = black_swan_lines_through_index(idx, y_top)

    return go.Scatter(
        x=swan_x,
        y=swan_y,
        mode="lines",
        line=dict(color=black_swan_color, width=2),
        opacity=0.82,
        name="",
        showlegend=False,
        customdata=swan_cd,
        hovertemplate=(
            "Black swan event<br>"
            "Date: %{customdata[0]}<br>"
            "Return: %{customdata[1]:.2f}%"
            "<extra></extra>"
        ),
    )


def make_left_distribution_traces(state, idx, y_top_left):
    pooled_line_cd = line_customdata(
        state["pooled_total_prob"],
        state["pooled_empirical_tail_rate"],
        state["pooled_n"],
        state["pooled_tail_count"],
    )

    pooled_left_x, pooled_left_y = filled_tail_area(
        state["pooled_curve"],
        side="left",
    )

    pooled_right_x, pooled_right_y = filled_tail_area(
        state["pooled_curve"],
        side="right",
    )

    pooled_left_cd = tail_customdata(
        pooled_left_x,
        state["pooled_left_prob"],
        state["pooled_empirical_tail_rate"],
        state["pooled_n"],
        state["pooled_tail_count"],
    )

    pooled_right_cd = tail_customdata(
        pooled_right_x,
        state["pooled_right_prob"],
        state["pooled_empirical_tail_rate"],
        state["pooled_n"],
        state["pooled_tail_count"],
    )

    lower_line_x, lower_line_y = vertical_line(lower_tail_cutoff, y_top_left)
    upper_line_x, upper_line_y = vertical_line(upper_tail_cutoff, y_top_left)

    return [
        go.Bar(
            x=bin_centers,
            y=state["histogram_counts"],
            width=BIN_WIDTH_PERCENT * 0.92,
            marker=dict(color=histogram_color),
            opacity=0.72,
            name="",
            showlegend=False,
            hovertemplate=(
                "Return bucket: %{x:.2f}%<br>"
                "Observations: %{y:,}"
                "<extra></extra>"
            ),
        ),
        go.Scatter(
            x=x_grid,
            y=state["pooled_curve"],
            mode="lines",
            line=dict(color=pooled_color, width=3),
            name="",
            showlegend=False,
            customdata=pooled_line_cd,
            hovertemplate=(
                "Return: %{x:.2f}%<br>"
                "Expected bin count: %{y:,.2f}<br>"
                "Parametric tail probability: %{customdata[0]:.6%}<br>"
                "Empirical tail rate: %{customdata[1]:.6%}<br>"
                "Observations: %{customdata[2]:,.0f}<br>"
                "Tail events: %{customdata[3]:,.0f}"
                "<extra></extra>"
            ),
        ),
        go.Scatter(
            x=pooled_left_x,
            y=pooled_left_y,
            mode="lines",
            line=dict(width=0),
            fill="toself",
            fillcolor=pooled_tail_fill,
            name="",
            showlegend=False,
            customdata=pooled_left_cd,
            hovertemplate=(
                "Left tail probability: %{customdata[0]:.6%}<br>"
                "Empirical total tail rate: %{customdata[1]:.6%}"
                "<extra></extra>"
            ),
        ),
        go.Scatter(
            x=pooled_right_x,
            y=pooled_right_y,
            mode="lines",
            line=dict(width=0),
            fill="toself",
            fillcolor=pooled_tail_fill,
            name="",
            showlegend=False,
            customdata=pooled_right_cd,
            hovertemplate=(
                "Right tail probability: %{customdata[0]:.6%}<br>"
                "Empirical total tail rate: %{customdata[1]:.6%}"
                "<extra></extra>"
            ),
        ),
        make_black_swan_trace(idx, y_top_left),
        go.Scatter(
            x=lower_line_x,
            y=lower_line_y,
            mode="lines",
            line=dict(color=cutoff_color, width=1.5, dash="dot"),
            name="",
            showlegend=False,
            hoverinfo="skip",
        ),
        go.Scatter(
            x=upper_line_x,
            y=upper_line_y,
            mode="lines",
            line=dict(color=cutoff_color, width=1.5, dash="dot"),
            name="",
            showlegend=False,
            hoverinfo="skip",
        ),
    ]


def make_right_distribution_traces(state, idx, y_top_right):
    traces = []

    for regime in REGIME_ORDER:
        regime_state = state["regimes"][regime]

        regime_line_cd = line_customdata(
            regime_state["total_prob"],
            regime_state["empirical_tail_rate"],
            regime_state["n"],
            regime_state["tail_count"],
        )

        left_x, left_y = filled_tail_area(
            regime_state["curve"],
            side="left",
        )

        right_x, right_y = filled_tail_area(
            regime_state["curve"],
            side="right",
        )

        left_cd = tail_customdata(
            left_x,
            regime_state["left_prob"],
            regime_state["empirical_tail_rate"],
            regime_state["n"],
            regime_state["tail_count"],
        )

        right_cd = tail_customdata(
            right_x,
            regime_state["right_prob"],
            regime_state["empirical_tail_rate"],
            regime_state["n"],
            regime_state["tail_count"],
        )

        traces.extend(
            [
                go.Scatter(
                    x=x_grid,
                    y=regime_state["curve"],
                    mode="lines",
                    line=dict(color=regime_colors[regime], width=3),
                    name="",
                    showlegend=False,
                    customdata=regime_line_cd,
                    hovertemplate=(
                        f"{regime}<br>"
                        "Return: %{x:.2f}%<br>"
                        "Expected bin count: %{y:,.2f}<br>"
                        "Conditional parametric tail probability: %{customdata[0]:.6%}<br>"
                        "Empirical regime tail rate: %{customdata[1]:.6%}<br>"
                        "Regime observations: %{customdata[2]:,.0f}<br>"
                        "Regime tail events: %{customdata[3]:,.0f}"
                        "<extra></extra>"
                    ),
                ),
                go.Scatter(
                    x=left_x,
                    y=left_y,
                    mode="lines",
                    line=dict(width=0),
                    fill="toself",
                    fillcolor=regime_fills[regime],
                    name="",
                    showlegend=False,
                    customdata=left_cd,
                    hovertemplate=(
                        f"{regime}<br>"
                        "Left tail probability: %{customdata[0]:.6%}<br>"
                        "Empirical regime tail rate: %{customdata[1]:.6%}"
                        "<extra></extra>"
                    ),
                ),
                go.Scatter(
                    x=right_x,
                    y=right_y,
                    mode="lines",
                    line=dict(width=0),
                    fill="toself",
                    fillcolor=regime_fills[regime],
                    name="",
                    showlegend=False,
                    customdata=right_cd,
                    hovertemplate=(
                        f"{regime}<br>"
                        "Right tail probability: %{customdata[0]:.6%}<br>"
                        "Empirical regime tail rate: %{customdata[1]:.6%}"
                        "<extra></extra>"
                    ),
                ),
            ]
        )

    lower_line_x, lower_line_y = vertical_line(lower_tail_cutoff, y_top_right)
    upper_line_x, upper_line_y = vertical_line(upper_tail_cutoff, y_top_right)

    traces.extend(
        [
            make_black_swan_trace(idx, y_top_right),
            go.Scatter(
                x=lower_line_x,
                y=lower_line_y,
                mode="lines",
                line=dict(color=cutoff_color, width=1.5, dash="dot"),
                name="",
                showlegend=False,
                hoverinfo="skip",
            ),
            go.Scatter(
                x=upper_line_x,
                y=upper_line_y,
                mode="lines",
                line=dict(color=cutoff_color, width=1.5, dash="dot"),
                name="",
                showlegend=False,
                hoverinfo="skip",
            ),
        ]
    )

    return traces


# -----------------------------------------------------------------------------
# Helpers: volatility forecast panel
# -----------------------------------------------------------------------------

def forecast_garch_vol_from_index(idx: int):
    current_date = df_sub.loc[idx, "Date"]

    future_dates = pd.bdate_range(
        current_date + pd.offsets.BDay(1),
        periods=VOL_FORECAST_HORIZON_DAYS,
    )

    current_resid = float(df_sub.loc[idx, "garch_resid_pct"])
    current_var = float(df_sub.loc[idx, "garch_var_daily_pct2"])

    forecast_vars = []

    next_var = (
        garch_omega
        + garch_alpha * current_resid ** 2
        + garch_beta * current_var
    )

    forecast_vars.append(next_var)

    persistence = garch_alpha + garch_beta

    for _ in range(1, VOL_FORECAST_HORIZON_DAYS):
        next_var = garch_omega + persistence * next_var
        forecast_vars.append(next_var)

    forecast_vol = np.sqrt(np.maximum(forecast_vars, 0)) * ANNUALIZATION_FACTOR

    error_window_start = max(0, idx - VOL_FORECAST_BAND_LOOKBACK + 1)
    recent_error = vol_model_error.iloc[error_window_start : idx + 1].dropna()

    if len(recent_error) >= 20:
        forecast_band_std = recent_error.std(ddof=1)
    else:
        forecast_band_std = vol_model_error.dropna().std(ddof=1)

    if not np.isfinite(forecast_band_std):
        forecast_band_std = 0.0

    forecast_upper = forecast_vol + forecast_band_std
    forecast_lower = np.maximum(forecast_vol - forecast_band_std, 0)

    return future_dates, forecast_vol, forecast_lower, forecast_upper, forecast_band_std


def make_volatility_traces(idx: int):
    data = df_sub.iloc[: idx + 1].copy()

    current_regime = str(df_sub.loc[idx, "vol_regime"])

    history_traces = []

    for regime in REGIME_ORDER:
        y = data["realized_vol_ann_pct"].where(data["vol_regime"] == regime)

        history_traces.append(
            go.Scatter(
                x=data["Date"],
                y=y,
                mode="lines",
                line=dict(color=regime_colors[regime], width=2.4),
                name="",
                showlegend=False,
                connectgaps=False,
                hovertemplate=(
                    f"{regime}<br>"
                    "Date: %{x|%b %d, %Y}<br>"
                    "Realized volatility: %{y:.2f}%"
                    "<extra></extra>"
                ),
            )
        )

    future_dates, forecast_vol, forecast_lower, forecast_upper, forecast_band_std = (
        forecast_garch_vol_from_index(idx)
    )

    forecast_customdata = [
        [forecast_band_std, current_regime]
        for _ in range(len(future_dates))
    ]

    band_lower_trace = go.Scatter(
        x=future_dates,
        y=forecast_lower,
        mode="lines",
        line=dict(width=0),
        name="",
        showlegend=False,
        hoverinfo="skip",
    )

    band_upper_trace = go.Scatter(
        x=future_dates,
        y=forecast_upper,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=forecast_band_opacity[current_regime],
        name="",
        showlegend=False,
        customdata=forecast_customdata,
        hovertemplate=(
            "Date: %{x|%b %d, %Y}<br>"
            "Upper band: %{y:.2f}%<br>"
            "Forecast σ band: %{customdata[0]:.2f}%<br>"
            "Current regime: %{customdata[1]}"
            "<extra></extra>"
        ),
    )

    forecast_trace = go.Scatter(
        x=future_dates,
        y=forecast_vol,
        mode="lines",
        line=dict(color=regime_colors[current_regime], width=3, dash="dash"),
        name="",
        showlegend=False,
        customdata=forecast_customdata,
        hovertemplate=(
            "Date: %{x|%b %d, %Y}<br>"
            "Expected volatility: %{y:.2f}%<br>"
            "Forecast σ band: %{customdata[0]:.2f}%<br>"
            "Current regime: %{customdata[1]}"
            "<extra></extra>"
        ),
    )

    current_marker = go.Scatter(
        x=[df_sub.loc[idx, "Date"]],
        y=[df_sub.loc[idx, "realized_vol_ann_pct"]],
        mode="markers",
        marker=dict(
            color=regime_colors[current_regime],
            size=9,
            line=dict(color=off_white, width=1),
        ),
        name="",
        showlegend=False,
        customdata=[[current_regime]],
        hovertemplate=(
            "Date: %{x|%b %d, %Y}<br>"
            "Realized volatility: %{y:.2f}%<br>"
            "Current regime: %{customdata[0]}"
            "<extra></extra>"
        ),
    )

    return history_traces + [
        band_lower_trace,
        band_upper_trace,
        forecast_trace,
        current_marker,
    ]


# -----------------------------------------------------------------------------
# Precompute states and y-axis ranges
# -----------------------------------------------------------------------------

distribution_states = {}
volatility_traces_by_year = {}

left_y_candidates = []
right_y_candidates = []
vol_y_candidates = []

for year in years:
    idx = year_to_end_idx[year]

    state = build_distribution_state(idx)
    distribution_states[year] = state

    left_y_candidates.append(np.max(state["histogram_counts"]))
    left_y_candidates.append(np.max(state["pooled_curve"]))

    for regime in REGIME_ORDER:
        right_y_candidates.append(np.max(state["regimes"][regime]["curve"]))

    vol_traces = make_volatility_traces(idx)
    volatility_traces_by_year[year] = vol_traces

    for trace in vol_traces:
        y_values = np.asarray(trace.y, dtype=float)
        finite_y_values = y_values[np.isfinite(y_values)]
        if len(finite_y_values):
            vol_y_candidates.append(np.max(finite_y_values))

left_y_max = max(
    5,
    int(np.ceil(max(left_y_candidates) * max(HISTOGRAM_Y_PADDING, CURVE_Y_PADDING))),
)

right_y_max = max(
    5,
    int(np.ceil(max(right_y_candidates) * CURVE_Y_PADDING)),
)

vol_y_max = max(
    5,
    float(np.ceil(max(vol_y_candidates) * VOL_Y_PADDING)),
)


# -----------------------------------------------------------------------------
# Initial state
# -----------------------------------------------------------------------------

initial_year = years[0]
initial_idx = year_to_end_idx[initial_year]
initial_distribution_state = distribution_states[initial_year]

initial_left_traces = make_left_distribution_traces(
    initial_distribution_state,
    initial_idx,
    left_y_max,
)

initial_right_traces = make_right_distribution_traces(
    initial_distribution_state,
    initial_idx,
    right_y_max,
)

initial_vol_traces = volatility_traces_by_year[initial_year]

initial_traces = initial_left_traces + initial_right_traces + initial_vol_traces


# -----------------------------------------------------------------------------
# Build figure
# -----------------------------------------------------------------------------

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{}, {}],
        [{"colspan": 2}, None],
    ],
    row_heights=[0.62, 0.38],
    vertical_spacing=0.115,
    horizontal_spacing=0.075,
    shared_yaxes=False,
)

# Top-left panel: pooled calibrated normal distribution
for trace in initial_left_traces:
    fig.add_trace(trace, row=1, col=1)

# Top-right panel: conditional normal distributions by GARCH regime
for trace in initial_right_traces:
    fig.add_trace(trace, row=1, col=2)

# Bottom panel: realized volatility plus forward forecast
for trace in initial_vol_traces:
    fig.add_trace(trace, row=2, col=1)

# Zero-return baselines on top panels
fig.add_vline(
    x=0,
    row=1,
    col=1,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.70,
)

fig.add_vline(
    x=0,
    row=1,
    col=2,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.70,
)


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
slider_steps = []

trace_indices = list(range(len(initial_traces)))

for year in years:
    idx = year_to_end_idx[year]
    frame_name = f"year_{year}"

    state = distribution_states[year]

    frame_left_traces = make_left_distribution_traces(
        state,
        idx,
        left_y_max,
    )

    frame_right_traces = make_right_distribution_traces(
        state,
        idx,
        right_y_max,
    )

    frame_vol_traces = volatility_traces_by_year[year]

    frame_traces = frame_left_traces + frame_right_traces + frame_vol_traces

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_traces,
            traces=trace_indices,
            layout=go.Layout(
                annotations=[wait_time_annotation_for_state(state)]
            ),
        )
    )

    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {
                        "duration": FRAME_DURATION_MS,
                        "redraw": True,
                    },
                    "transition": {
                        "duration": TRANSITION_DURATION_MS,
                        "easing": "cubic-in-out",
                    },
                    "mode": "immediate",
                    "fromcurrent": True,
                },
            ],
            "label": str(year),
            "method": "animate",
        }
    )

fig.frames = frames


# -----------------------------------------------------------------------------
# Layout
# -----------------------------------------------------------------------------

last_date = df_sub["Date"].max()
bottom_x_max = last_date + pd.offsets.BDay(
    VOL_FORECAST_HORIZON_DAYS + VOL_FORECAST_X_BUFFER_DAYS
)

fig.update_layout(
    title=dict(
        text="Modeling Conditional Regimes",
        x=0.5,
        xanchor="center",
        font=dict(color=off_white, size=25),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=900,
    width=1200,
    margin=dict(t=75, b=140, r=60, l=75),
    bargap=0.02,
    hovermode="closest",
    showlegend=False,
    annotations=[wait_time_annotation_for_state(initial_distribution_state)],
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": FRAME_DURATION_MS,
                                "redraw": True,
                            },
                            "transition": {
                                "duration": TRANSITION_DURATION_MS,
                                "easing": "cubic-in-out",
                            },
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                                "redraw": True,
                            },
                            "transition": {
                                "duration": 0,
                            },
                            "mode": "immediate",
                            "fromcurrent": True,
                        },
                    ],
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {
                    "size": 14,
                    "color": off_white,
                },
                "prefix": "Year: ",
                "visible": True,
                "xanchor": "right",
            },
            "transition": {
                "duration": 0,
            },
            "pad": {
                "b": 40,
                "t": 30,
            },
            "len": 0.85,
            "x": 0.15,
            "y": -0.055,
            "steps": slider_steps,
        }
    ],
)


# -----------------------------------------------------------------------------
# Axes
# -----------------------------------------------------------------------------

fig.update_xaxes(
    axis_style,
    range=[x_min, x_max],
    title_text="Daily Return",
    ticksuffix="%",
    tickformat=",.2f",
    row=1,
    col=1,
)

fig.update_xaxes(
    axis_style,
    range=[x_min, x_max],
    title_text="Daily Return",
    ticksuffix="%",
    tickformat=",.2f",
    row=1,
    col=2,
)

# Fixed full-history bottom x-axis behavior.
fig.update_xaxes(
    axis_style,
    range=[df_sub["Date"].min(), bottom_x_max],
    title_text="Date",
    row=2,
    col=1,
)

fig.update_yaxes(
    axis_style,
    range=[0, left_y_max],
    title_text="Frequency / Expected Bin Count",
    tickformat=",d",
    row=1,
    col=1,
)

fig.update_yaxes(
    axis_style,
    range=[0, right_y_max],
    title_text="Expected Bin Count",
    tickformat=",d",
    row=1,
    col=2,
)

fig.update_yaxes(
    axis_style,
    range=[0, vol_y_max],
    title_text="Annualized Realized Volatility",
    ticksuffix="%",
    tickformat=",.1f",
    row=2,
    col=1,
)

fig.show()

---

##### 🔮 Failure of Walk-Forward Empirical Modeling

So let's just use empirical probabilities then, right?

They *fail* - no amount of walk forward testing is useful here, no amount of statistics can predict the future.

The purpose of modeling is not prediction, it's to ensure effective positioning and survival.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from arch import arch_model
except ImportError as exc:
    raise ImportError(
        "This snippet requires the `arch` package. Install it with: pip install arch"
    ) from exc


# =============================================================================
# Walk-forward validation failure demo
# Top panel: equity curve from frozen 2001 empirical VaR/CVaR sizing.
# Bottom panel: GARCH realized-volatility regime animation and forward forecast.
#
# Interpretation:
# - We proxy a short-option / VRP harvest book as a short-variance-like payoff.
# - The model calibrates empirical VaR/CVaR by GARCH regime using only data
#   available through 2001.
# - It then blindly sizes positions from 2002 onward using those frozen 2001
#   loss estimates.
# - In 2008, realized variance is far outside the frozen empirical loss map,
#   so the strategy blows out.
#
# This is an empirical-modeling illustration, not a production option-pricing
# engine. Tune the risk and payoff knobs below to make the failure more or less
# dramatic for your exact SPY CSV.
# =============================================================================


# -----------------------------------------------------------------------------
# Load SPY data
# -----------------------------------------------------------------------------

FILE = "spy_1999_2026.csv"

df = pd.read_csv(FILE)

date_column = [c for c in df.columns if c.strip().lower() == "date"][0]

close_column = None
for c in df.columns:
    normalized = c.lower().replace(" ", "")
    if normalized in ["adjclose", "adjustedclose"]:
        close_column = c
        break

if close_column is None:
    close_column = [c for c in df.columns if c.strip().lower() == "close"][0]

df = df[[date_column, close_column]].copy()
df.rename(columns={date_column: "Date", close_column: "Close"}, inplace=True)

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df["Return"] = df["Close"].pct_change()


# -----------------------------------------------------------------------------
# Parameters
# -----------------------------------------------------------------------------

ANIMATION_START = "1999-01-01"
ANIMATION_END = "2009-12-31"

# This is the key failure mechanism:
# the risk map is frozen at the end of 2001 and used blindly afterward.
CALIBRATION_END = "2001-12-31"

INITIAL_EQUITY = 1_000_000.0

# Empirical VaR / CVaR level used for the frozen 2001 regime sizing.
VAR_ALPHA = 0.99

# Daily loss budget expressed as fraction of current equity.
# This is intentionally aggressive to make the empirical-model failure visible.
TARGET_ES_FRACTION_OF_EQUITY = 0.35

# Hard cap on notional leverage.
MAX_NOTIONAL_LEVERAGE = 120.0

# Numerical floor so tiny empirical losses do not produce infinite sizing.
MIN_EMPIRICAL_LOSS_FLOOR = 0.0010

# Short-option / VRP proxy.
# Unit P&L is per $1 notional:
#
#   unit_pnl = premium collected from implied variance
#              - realized variance charge
#
# This behaves like a stylized delta-hedged short option / short variance book.
# Increasing REALIZED_VARIANCE_LOSS_MULTIPLIER or TARGET_ES_FRACTION_OF_EQUITY
# makes the 2008 blowout more dramatic.
VARIANCE_PREMIUM_MULTIPLIER = 5.0
REALIZED_VARIANCE_LOSS_MULTIPLIER = 4.0

GARCH_P = 1
GARCH_Q = 1

REGIME_ORDER = ["Low Vol", "Med Vol", "High Vol"]

REALIZED_VOL_WINDOW = 21
ANNUALIZATION_FACTOR = np.sqrt(252)

VOL_FORECAST_HORIZON_DAYS = 63
VOL_FORECAST_BAND_LOOKBACK = 252
VOL_FORECAST_X_BUFFER_DAYS = 5

FRAME_DURATION_MS = 450
TRANSITION_DURATION_MS = 150


# -----------------------------------------------------------------------------
# Clean data
# -----------------------------------------------------------------------------

df_sub = (
    df.loc[
        (df["Date"] >= ANIMATION_START) & (df["Date"] <= ANIMATION_END),
        ["Date", "Close", "Return"],
    ]
    .dropna(subset=["Date", "Close", "Return"])
    .sort_values("Date")
    .reset_index(drop=True)
)

df_sub["return_pct"] = df_sub["Return"].astype(float) * 100.0
df_sub["year"] = df_sub["Date"].dt.year

calibration_end_ts = pd.Timestamp(CALIBRATION_END)

calibration_mask = df_sub["Date"] <= calibration_end_ts
live_mask = df_sub["Date"] > calibration_end_ts

if calibration_mask.sum() < 250:
    raise ValueError(
        "Calibration sample is too small. Check ANIMATION_START, CALIBRATION_END, "
        "and your CSV date range."
    )


# -----------------------------------------------------------------------------
# GARCH volatility regimes
# Fit GARCH(1,1) on percent returns.
#
# To make the failure visually explicit, the regime cutoffs are calibrated only
# on data available through 2001 and then frozen.
# -----------------------------------------------------------------------------

garch_returns = df_sub["return_pct"].astype(float)

garch_model = arch_model(
    garch_returns,
    mean="Constant",
    vol="GARCH",
    p=GARCH_P,
    q=GARCH_Q,
    dist="normal",
    rescale=False,
)

garch_result = garch_model.fit(disp="off")

df_sub["garch_vol_daily_pct"] = np.asarray(garch_result.conditional_volatility)
df_sub["garch_vol_ann_pct"] = df_sub["garch_vol_daily_pct"] * ANNUALIZATION_FACTOR

garch_params = garch_result.params

garch_mu = float(garch_params.get("mu", garch_params.get("Const", 0.0)))
garch_omega = float(garch_params["omega"])
garch_alpha = float(garch_params[[p for p in garch_params.index if p.startswith("alpha")][0]])
garch_beta = float(garch_params[[p for p in garch_params.index if p.startswith("beta")][0]])

df_sub["garch_resid_pct"] = df_sub["return_pct"] - garch_mu
df_sub["garch_var_daily_pct2"] = df_sub["garch_vol_daily_pct"] ** 2

# Frozen 2001 regime cutoffs.
low_cut, high_cut = df_sub.loc[calibration_mask, "garch_vol_daily_pct"].quantile(
    [1 / 3, 2 / 3]
)

df_sub["vol_regime"] = np.select(
    [
        df_sub["garch_vol_daily_pct"] <= low_cut,
        df_sub["garch_vol_daily_pct"] <= high_cut,
    ],
    [
        "Low Vol",
        "Med Vol",
    ],
    default="High Vol",
)

df_sub["vol_regime"] = pd.Categorical(
    df_sub["vol_regime"],
    categories=REGIME_ORDER,
    ordered=True,
)

df_sub["vol_regime_str"] = df_sub["vol_regime"].astype(str)


# -----------------------------------------------------------------------------
# Realized volatility for bottom panel
# -----------------------------------------------------------------------------

df_sub["realized_vol_ann_pct"] = (
    df_sub["return_pct"]
    .rolling(REALIZED_VOL_WINDOW)
    .std(ddof=1)
    * ANNUALIZATION_FACTOR
)

vol_model_error = df_sub["realized_vol_ann_pct"] - df_sub["garch_vol_ann_pct"]


# -----------------------------------------------------------------------------
# Short-option / VRP harvesting proxy
# -----------------------------------------------------------------------------

# Convert GARCH daily volatility from percent to decimal.
df_sub["garch_sigma_daily_dec"] = df_sub["garch_vol_daily_pct"] / 100.0

# Premium collected from implied variance proxy.
df_sub["vrp_daily_credit_unit"] = (
    VARIANCE_PREMIUM_MULTIPLIER
    * df_sub["garch_sigma_daily_dec"] ** 2
)

# Loss from realized variance shock.
df_sub["vrp_realized_var_charge_unit"] = (
    REALIZED_VARIANCE_LOSS_MULTIPLIER
    * df_sub["Return"] ** 2
)

# Unit P&L per $1 notional.
df_sub["vrp_unit_pnl"] = (
    df_sub["vrp_daily_credit_unit"]
    - df_sub["vrp_realized_var_charge_unit"]
)

# Positive values are losses for VaR/CVaR estimation.
df_sub["vrp_unit_loss"] = -df_sub["vrp_unit_pnl"]


# -----------------------------------------------------------------------------
# Frozen 2001 empirical VaR/CVaR sizing by regime
# -----------------------------------------------------------------------------

def empirical_var_cvar(losses: pd.Series, alpha: float):
    losses = losses.dropna().astype(float)

    if len(losses) == 0:
        return np.nan, np.nan, np.nan, 0

    var = float(np.quantile(losses, alpha))
    tail_losses = losses.loc[losses >= var]

    if len(tail_losses) == 0:
        cvar = var
    else:
        cvar = float(tail_losses.mean())

    max_loss = float(losses.max())
    n = int(len(losses))

    return var, cvar, max_loss, n


calibration_data = df_sub.loc[calibration_mask].copy()

pooled_var, pooled_cvar, pooled_max_loss, pooled_n = empirical_var_cvar(
    calibration_data["vrp_unit_loss"],
    VAR_ALPHA,
)

risk_by_regime = {}

for regime in REGIME_ORDER:
    regime_losses = calibration_data.loc[
        calibration_data["vol_regime_str"] == regime,
        "vrp_unit_loss",
    ]

    var, cvar, max_loss, n = empirical_var_cvar(regime_losses, VAR_ALPHA)

    # Fallback to pooled calibration if a regime has too little data.
    if n < 30 or not np.isfinite(var) or not np.isfinite(cvar):
        var = pooled_var
        cvar = pooled_cvar
        max_loss = pooled_max_loss
        n = pooled_n
        source = "pooled fallback"
    else:
        source = "regime"

    empirical_max_expected_loss = max(
        var,
        cvar,
        MIN_EMPIRICAL_LOSS_FLOOR,
    )

    leverage = min(
        TARGET_ES_FRACTION_OF_EQUITY / empirical_max_expected_loss,
        MAX_NOTIONAL_LEVERAGE,
    )

    risk_by_regime[regime] = {
        "VaR": var,
        "CVaR": cvar,
        "MaxLoss": max_loss,
        "MaxExpLoss": empirical_max_expected_loss,
        "Leverage": leverage,
        "N": n,
        "Source": source,
    }


def regime_stat(regime, key):
    return risk_by_regime[str(regime)][key]


df_sub["frozen_var_unit"] = [
    regime_stat(regime, "VaR") for regime in df_sub["vol_regime_str"]
]

df_sub["frozen_cvar_unit"] = [
    regime_stat(regime, "CVaR") for regime in df_sub["vol_regime_str"]
]

df_sub["frozen_max_exp_loss_unit"] = [
    regime_stat(regime, "MaxExpLoss") for regime in df_sub["vol_regime_str"]
]

df_sub["frozen_leverage"] = [
    regime_stat(regime, "Leverage") for regime in df_sub["vol_regime_str"]
]


# -----------------------------------------------------------------------------
# Equity simulation
# -----------------------------------------------------------------------------

equity_values = []
portfolio_returns = []
active_leverage_values = []

equity = INITIAL_EQUITY
blown_out = False
blowout_idx = None

for i, row in df_sub.iterrows():
    current_date = row["Date"]

    if i == 0:
        equity_values.append(equity)
        portfolio_returns.append(0.0)
        active_leverage_values.append(0.0)
        continue

    if current_date <= calibration_end_ts or blown_out:
        equity_values.append(equity)
        portfolio_returns.append(0.0)
        active_leverage_values.append(0.0)
        continue

    leverage = float(row["frozen_leverage"])
    unit_pnl = float(row["vrp_unit_pnl"])

    portfolio_return = leverage * unit_pnl
    new_equity = equity * (1.0 + portfolio_return)

    equity = new_equity

    equity_values.append(equity)
    portfolio_returns.append(portfolio_return)
    active_leverage_values.append(leverage)

    if equity <= 0 and blowout_idx is None:
        blown_out = True
        blowout_idx = i

df_sub["equity"] = equity_values
df_sub["portfolio_return"] = portfolio_returns
df_sub["active_leverage"] = active_leverage_values

df_sub["running_peak_equity"] = df_sub["equity"].cummax()
df_sub["drawdown"] = df_sub["equity"] / df_sub["running_peak_equity"] - 1.0


if blowout_idx is not None:
    blowout_date = df_sub.loc[blowout_idx, "Date"]
    blowout_equity = df_sub.loc[blowout_idx, "equity"]
else:
    blowout_date = None
    blowout_equity = None
    print(
        "No blowout occurred under the current knobs. "
        "Increase TARGET_ES_FRACTION_OF_EQUITY, MAX_NOTIONAL_LEVERAGE, "
        "or REALIZED_VARIANCE_LOSS_MULTIPLIER if you want a more dramatic failure."
    )


# -----------------------------------------------------------------------------
# Animation frame dates: one frame per year, 1999 through 2009
# -----------------------------------------------------------------------------

years = df_sub["year"].drop_duplicates().to_list()

year_end_indices = (
    df_sub.groupby("year", sort=True)
    .tail(1)
    .index
    .to_numpy()
)

year_to_end_idx = dict(zip(years, year_end_indices))


# -----------------------------------------------------------------------------
# Styling
# -----------------------------------------------------------------------------

off_white = "#e0e0e0"

equity_color = "#00d4ff"
equity_marker_color = "#ffffff"
equity_zero_color = "#ff3b30"
equity_baseline_color = "#777777"
calibration_fill = "rgba(255,170,51,0.10)"
blowout_color = "#ff3b30"

forecast_band_opacity = {
    "Low Vol": "rgba(95,208,255,0.16)",
    "Med Vol": "rgba(255,170,51,0.16)",
    "High Vol": "rgba(242,91,55,0.18)",
}

regime_colors = {
    "Low Vol": "#5fd0ff",
    "Med Vol": "#ffaa33",
    "High Vol": "#f25b37",
}

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)


# -----------------------------------------------------------------------------
# Formatting helpers
# -----------------------------------------------------------------------------

def fmt_money(x):
    if not np.isfinite(x):
        return "n/a"
    return f"${x:,.0f}"


def fmt_pct(x):
    if not np.isfinite(x):
        return "n/a"
    return f"{x:.2%}"


def fmt_unit_loss(x):
    if not np.isfinite(x):
        return "n/a"
    return f"{x:.4%}"


# -----------------------------------------------------------------------------
# GARCH forecast helpers
# -----------------------------------------------------------------------------

def forecast_garch_vol_from_index(idx: int):
    current_date = df_sub.loc[idx, "Date"]

    future_dates = pd.bdate_range(
        current_date + pd.offsets.BDay(1),
        periods=VOL_FORECAST_HORIZON_DAYS,
    )

    current_resid = float(df_sub.loc[idx, "garch_resid_pct"])
    current_var = float(df_sub.loc[idx, "garch_var_daily_pct2"])

    forecast_vars = []

    next_var = (
        garch_omega
        + garch_alpha * current_resid ** 2
        + garch_beta * current_var
    )

    forecast_vars.append(next_var)

    persistence = garch_alpha + garch_beta

    for _ in range(1, VOL_FORECAST_HORIZON_DAYS):
        next_var = garch_omega + persistence * next_var
        forecast_vars.append(next_var)

    forecast_vol = np.sqrt(np.maximum(forecast_vars, 0)) * ANNUALIZATION_FACTOR

    error_window_start = max(0, idx - VOL_FORECAST_BAND_LOOKBACK + 1)
    recent_error = vol_model_error.iloc[error_window_start : idx + 1].dropna()

    if len(recent_error) >= 20:
        forecast_band_std = recent_error.std(ddof=1)
    else:
        forecast_band_std = vol_model_error.dropna().std(ddof=1)

    if not np.isfinite(forecast_band_std):
        forecast_band_std = 0.0

    forecast_upper = forecast_vol + forecast_band_std
    forecast_lower = np.maximum(forecast_vol - forecast_band_std, 0)

    return future_dates, forecast_vol, forecast_lower, forecast_upper, forecast_band_std


# -----------------------------------------------------------------------------
# Trace builders
# -----------------------------------------------------------------------------

def make_equity_traces(idx: int):
    data = df_sub.iloc[: idx + 1].copy()

    equity_customdata = data[
        [
            "vol_regime_str",
            "active_leverage",
            "portfolio_return",
            "vrp_unit_pnl",
            "frozen_var_unit",
            "frozen_cvar_unit",
            "frozen_max_exp_loss_unit",
        ]
    ].to_numpy()

    equity_trace = go.Scatter(
        x=data["Date"],
        y=data["equity"],
        mode="lines",
        line=dict(color=equity_color, width=3.2),
        name="Frozen 2001 VaR/CVaR equity",
        showlegend=False,
        customdata=equity_customdata,
        hovertemplate=(
            "Date: %{x|%b %d, %Y}<br>"
            "Equity: $%{y:,.0f}<br>"
            "Regime: %{customdata[0]}<br>"
            "Active leverage: %{customdata[1]:.1f}x<br>"
            "Portfolio return: %{customdata[2]:.2%}<br>"
            "Unit VRP P&L: %{customdata[3]:.4%}<br>"
            "Frozen VaR: %{customdata[4]:.4%}<br>"
            "Frozen CVaR: %{customdata[5]:.4%}<br>"
            "Frozen max exp loss: %{customdata[6]:.4%}"
            "<extra></extra>"
        ),
    )

    current = df_sub.loc[idx]

    current_marker = go.Scatter(
        x=[current["Date"]],
        y=[current["equity"]],
        mode="markers",
        marker=dict(
            color=equity_marker_color,
            size=9,
            line=dict(color=equity_color, width=2),
        ),
        name="Current equity",
        showlegend=False,
        hovertemplate=(
            "Current frame<br>"
            "Date: %{x|%b %d, %Y}<br>"
            "Equity: $%{y:,.0f}"
            "<extra></extra>"
        ),
    )

    if blowout_idx is not None and blowout_idx <= idx:
        blowout_marker = go.Scatter(
            x=[df_sub.loc[blowout_idx, "Date"]],
            y=[df_sub.loc[blowout_idx, "equity"]],
            mode="markers+text",
            marker=dict(
                color=blowout_color,
                size=15,
                symbol="x",
                line=dict(color=off_white, width=2),
            ),
            text=["BLOWOUT"],
            textposition="top center",
            textfont=dict(color=blowout_color, size=13),
            name="Blowout",
            showlegend=False,
            hovertemplate=(
                "Account blowout<br>"
                "Date: %{x|%b %d, %Y}<br>"
                "Equity: $%{y:,.0f}"
                "<extra></extra>"
            ),
        )
    else:
        blowout_marker = go.Scatter(
            x=[],
            y=[],
            mode="markers+text",
            marker=dict(color=blowout_color, size=15, symbol="x"),
            text=[],
            name="Blowout",
            showlegend=False,
            hoverinfo="skip",
        )

    return [equity_trace, current_marker, blowout_marker]


def make_volatility_traces(idx: int):
    data = df_sub.iloc[: idx + 1].copy()
    current_regime = str(df_sub.loc[idx, "vol_regime"])

    history_traces = []

    for regime in REGIME_ORDER:
        y = data["realized_vol_ann_pct"].where(data["vol_regime_str"] == regime)

        history_traces.append(
            go.Scatter(
                x=data["Date"],
                y=y,
                mode="lines",
                line=dict(color=regime_colors[regime], width=2.4),
                name="",
                showlegend=False,
                connectgaps=False,
                hovertemplate=(
                    f"{regime}<br>"
                    "Date: %{x|%b %d, %Y}<br>"
                    "Realized volatility: %{y:.2f}%"
                    "<extra></extra>"
                ),
            )
        )

    future_dates, forecast_vol, forecast_lower, forecast_upper, forecast_band_std = (
        forecast_garch_vol_from_index(idx)
    )

    forecast_customdata = [
        [forecast_band_std, current_regime]
        for _ in range(len(future_dates))
    ]

    band_lower_trace = go.Scatter(
        x=future_dates,
        y=forecast_lower,
        mode="lines",
        line=dict(width=0),
        name="",
        showlegend=False,
        hoverinfo="skip",
    )

    band_upper_trace = go.Scatter(
        x=future_dates,
        y=forecast_upper,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=forecast_band_opacity[current_regime],
        name="",
        showlegend=False,
        customdata=forecast_customdata,
        hovertemplate=(
            "Date: %{x|%b %d, %Y}<br>"
            "Upper band: %{y:.2f}%<br>"
            "Forecast σ band: %{customdata[0]:.2f}%<br>"
            "Current regime: %{customdata[1]}"
            "<extra></extra>"
        ),
    )

    forecast_trace = go.Scatter(
        x=future_dates,
        y=forecast_vol,
        mode="lines",
        line=dict(color=regime_colors[current_regime], width=3, dash="dash"),
        name="",
        showlegend=False,
        customdata=forecast_customdata,
        hovertemplate=(
            "Date: %{x|%b %d, %Y}<br>"
            "Expected volatility: %{y:.2f}%<br>"
            "Forecast σ band: %{customdata[0]:.2f}%<br>"
            "Current regime: %{customdata[1]}"
            "<extra></extra>"
        ),
    )

    current_marker = go.Scatter(
        x=[df_sub.loc[idx, "Date"]],
        y=[df_sub.loc[idx, "realized_vol_ann_pct"]],
        mode="markers",
        marker=dict(
            color=regime_colors[current_regime],
            size=9,
            line=dict(color=off_white, width=1),
        ),
        name="",
        showlegend=False,
        customdata=[[current_regime]],
        hovertemplate=(
            "Date: %{x|%b %d, %Y}<br>"
            "Realized volatility: %{y:.2f}%<br>"
            "Current regime: %{customdata[0]}"
            "<extra></extra>"
        ),
    )

    return history_traces + [
        band_lower_trace,
        band_upper_trace,
        forecast_trace,
        current_marker,
    ]


# -----------------------------------------------------------------------------
# Annotation builders
# -----------------------------------------------------------------------------
def make_equity_annotation(idx: int):
    row = df_sub.loc[idx]

    current_date = row["Date"]
    current_regime = row["vol_regime_str"]

    if current_date <= calibration_end_ts:
        status = (
            "<span style='color:#ffaa33'><b>Calibration period</b></span><br>"
            "No live trading yet. The frozen risk map is being learned."
        )
    elif blowout_idx is not None and idx >= blowout_idx:
        status = (
            "<span style='color:#ff3b30'><b>Account blowout</b></span><br>"
            f"Frozen 2001 empirical tails failed in {blowout_date:%Y}."
        )
    else:
        status = (
            "<span style='color:#00d4ff'><b>Live blind sizing</b></span><br>"
            "Position size still uses only the frozen 2001 VaR/CVaR map."
        )

    text = (
        "<b>Frozen 2001 VaR/CVaR VRP sizing</b><br>"
        f"{status}<br>"
        f"Date: {current_date:%b %d, %Y}<br>"
        f"Equity: {fmt_money(row['equity'])}<br>"
        f"Regime: <span style='color:{regime_colors[current_regime]}'><b>{current_regime}</b></span><br>"
        f"Active leverage: {row['active_leverage']:.1f}x<br>"
        f"Frozen VaR: {fmt_unit_loss(row['frozen_var_unit'])}<br>"
        f"Frozen CVaR: {fmt_unit_loss(row['frozen_cvar_unit'])}<br>"
        f"Max exp loss used: {fmt_unit_loss(row['frozen_max_exp_loss_unit'])}<br>"
        f"Daily portfolio return: {fmt_pct(row['portfolio_return'])}"
    )

    # Place in upper left of first chart instead of upper right
    return dict(
        x=0.015,
        y=0.965,
        xref="paper",
        yref="y domain",
        xanchor="left",
        yanchor="top",
        align="left",
        text=text,
        showarrow=False,
        bgcolor="rgba(10,10,10,0.74)",
        bordercolor="rgba(255,255,255,0.22)",
        borderwidth=1,
        borderpad=7,
        font=dict(color=off_white, size=11),
    )


def make_garch_annotation(idx: int):
    row = df_sub.loc[idx]
    current_regime = row["vol_regime_str"]

    text = (
        "<b>GARCH regime monitor</b><br>"
        f"Current regime: "
        f"<span style='color:{regime_colors[current_regime]}'><b>{current_regime}</b></span><br>"
        f"GARCH daily vol: {row['garch_vol_daily_pct']:.2f}%<br>"
        f"GARCH annual vol: {row['garch_vol_ann_pct']:.2f}%<br>"
        f"21D realized vol: {row['realized_vol_ann_pct']:.2f}%<br>"
        "<br>"
        "<b>Frozen 2001 regime cutoffs</b><br>"
        f"Low ≤ {low_cut:.2f}% daily σ<br>"
        f"High &gt; {high_cut:.2f}% daily σ"
    )

    # Place in upper left of second chart instead of upper right
    return dict(
        x=0.015,
        y=0.965,
        xref="paper",
        yref="y2 domain",
        xanchor="left",
        yanchor="top",
        align="left",
        text=text,
        showarrow=False,
        bgcolor="rgba(10,10,10,0.74)",
        bordercolor="rgba(255,255,255,0.22)",
        borderwidth=1,
        borderpad=7,
        font=dict(color=off_white, size=11),
    )


def make_frame_annotations(idx: int):
    return [
        make_equity_annotation(idx),
        make_garch_annotation(idx),
    ]


# -----------------------------------------------------------------------------
# Precompute volatility y range
# -----------------------------------------------------------------------------

vol_y_candidates = []

volatility_traces_by_year = {}

for year in years:
    idx = year_to_end_idx[year]
    vol_traces = make_volatility_traces(idx)
    volatility_traces_by_year[year] = vol_traces

    for trace in vol_traces:
        y_values = np.asarray(trace.y, dtype=float)
        finite_y_values = y_values[np.isfinite(y_values)]
        if len(finite_y_values):
            vol_y_candidates.append(np.max(finite_y_values))

vol_y_max = max(
    5,
    float(np.ceil(max(vol_y_candidates) * 1.18)),
)

equity_min = min(float(df_sub["equity"].min()), 0.0)
equity_max = max(float(df_sub["equity"].max()), INITIAL_EQUITY)

equity_range = equity_max - equity_min
if equity_range <= 0:
    equity_range = INITIAL_EQUITY

equity_y_min = equity_min - 0.08 * equity_range
equity_y_max = equity_max + 0.10 * equity_range


# -----------------------------------------------------------------------------
# Initial traces
# -----------------------------------------------------------------------------

initial_year = years[0]
initial_idx = year_to_end_idx[initial_year]

initial_equity_traces = make_equity_traces(initial_idx)
initial_vol_traces = volatility_traces_by_year[initial_year]

initial_traces = initial_equity_traces + initial_vol_traces


# -----------------------------------------------------------------------------
# Build figure
# -----------------------------------------------------------------------------

fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.47, 0.53],
    vertical_spacing=0.105,
    shared_xaxes=False,
)

# Top panel: equity curve
for trace in initial_equity_traces:
    fig.add_trace(trace, row=1, col=1)

# Bottom panel: GARCH vol animation
for trace in initial_vol_traces:
    fig.add_trace(trace, row=2, col=1)


# -----------------------------------------------------------------------------
# Static reference shapes
# -----------------------------------------------------------------------------

# Calibration region on top equity panel.
fig.add_vrect(
    x0=pd.Timestamp(ANIMATION_START),
    x1=calibration_end_ts,
    row=1,
    col=1,
    fillcolor=calibration_fill,
    line_width=0,
    layer="below",
)

# Initial equity baseline.
fig.add_hline(
    y=INITIAL_EQUITY,
    row=1,
    col=1,
    line=dict(color=equity_baseline_color, width=1, dash="dash"),
    opacity=0.7,
)

# Zero equity line.
fig.add_hline(
    y=0,
    row=1,
    col=1,
    line=dict(color=equity_zero_color, width=2, dash="dot"),
    opacity=0.9,
)

# Calibration cutoff.
fig.add_vline(
    x=calibration_end_ts,
    row=1,
    col=1,
    line=dict(color="#ffaa33", width=1.5, dash="dot"),
    opacity=0.85,
)


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
slider_steps = []

trace_indices = list(range(len(initial_traces)))

for year in years:
    idx = year_to_end_idx[year]
    frame_name = f"year_{year}"

    frame_equity_traces = make_equity_traces(idx)
    frame_vol_traces = volatility_traces_by_year[year]

    frame_traces = frame_equity_traces + frame_vol_traces

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_traces,
            traces=trace_indices,
            layout=go.Layout(
                annotations=make_frame_annotations(idx),
            ),
        )
    )

    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {
                        "duration": FRAME_DURATION_MS,
                        "redraw": True,
                    },
                    "transition": {
                        "duration": TRANSITION_DURATION_MS,
                        "easing": "cubic-in-out",
                    },
                    "mode": "immediate",
                    "fromcurrent": True,
                },
            ],
            "label": str(year),
            "method": "animate",
        }
    )

fig.frames = frames


# -----------------------------------------------------------------------------
# Layout
# -----------------------------------------------------------------------------

last_date = df_sub["Date"].max()
bottom_x_max = last_date + pd.offsets.BDay(
    VOL_FORECAST_HORIZON_DAYS + VOL_FORECAST_X_BUFFER_DAYS
)

fig.update_layout(
    title=dict(
        text=(
            "Walk-Forward Validation Failure: "
            "Frozen 2001 VaR/CVaR Sizing Blows Out in 2008"
        ),
        x=0.5,
        xanchor="center",
        font=dict(color=off_white, size=24),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=900,
    width=1200,
    margin=dict(t=78, b=140, r=60, l=85),
    hovermode="closest",
    showlegend=False,
    annotations=make_frame_annotations(initial_idx),
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": FRAME_DURATION_MS,
                                "redraw": True,
                            },
                            "transition": {
                                "duration": TRANSITION_DURATION_MS,
                                "easing": "cubic-in-out",
                            },
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                                "redraw": True,
                            },
                            "transition": {
                                "duration": 0,
                            },
                            "mode": "immediate",
                            "fromcurrent": True,
                        },
                    ],
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {
                    "size": 14,
                    "color": off_white,
                },
                "prefix": "Year: ",
                "visible": True,
                "xanchor": "right",
            },
            "transition": {
                "duration": 0,
            },
            "pad": {
                "b": 40,
                "t": 30,
            },
            "len": 0.85,
            "x": 0.15,
            "y": -0.055,
            "steps": slider_steps,
        }
    ],
)


# -----------------------------------------------------------------------------
# Axes
# -----------------------------------------------------------------------------

fig.update_xaxes(
    axis_style,
    range=[pd.Timestamp(ANIMATION_START), pd.Timestamp(ANIMATION_END)],
    title_text="Date",
    row=1,
    col=1,
)

fig.update_yaxes(
    axis_style,
    range=[equity_y_min, equity_y_max],
    title_text="Equity",
    tickprefix="$",
    tickformat=",.0f",
    row=1,
    col=1,
)

fig.update_xaxes(
    axis_style,
    range=[pd.Timestamp(ANIMATION_START), bottom_x_max],
    title_text="Date",
    row=2,
    col=1,
)

fig.update_yaxes(
    axis_style,
    range=[0, vol_y_max],
    title_text="Annualized Realized Volatility",
    ticksuffix="%",
    tickformat=",.1f",
    row=2,
    col=1,
)


# -----------------------------------------------------------------------------
# Print frozen 2001 risk map for sanity checking
# -----------------------------------------------------------------------------

risk_table = (
    pd.DataFrame(risk_by_regime)
    .T[
        [
            "VaR",
            "CVaR",
            "MaxLoss",
            "MaxExpLoss",
            "Leverage",
            "N",
            "Source",
        ]
    ]
)

if blowout_idx is not None:
    print(
        f"\nBlowout date: {blowout_date:%Y-%m-%d} | "
        f"Equity: ${blowout_equity:,.0f}"
    )


fig.show()

# Optional export:
# fig.write_html(
#     "walk_forward_var_cvar_failure_1999_2009.html",
#     include_plotlyjs="cdn",
# )

---

##### 🏛️ Prediction vs. Positioning

Whenever we build a model we are not looking to *predict* anything.

Rather, we are aiming for effective statistics that ensure proper positioning and survival much like a casino.

If a big player comes into the casino and bets everything he has on red, the casino may not take the other side as no amount of edge can save them from the state of the world they lose that trade

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_vrt_daily_returns_10y.csv"

TRADING_DAYS_PER_YEAR = 252
SIM_YEARS = 10
SIM_DAYS = SIM_YEARS * TRADING_DAYS_PER_YEAR

INITIAL_INDEX_VALUE = 100

N_SIM_PATHS = 5000
N_DISPLAY_PATHS = 8

SEED = 42

FRAME_STRIDE = 21
FRAME_DURATION = 30
ANIMATION_REDRAW = False

RISK_FREE_RATE = 0.02

HOT_STOCK_BETA = 2.5
HOT_STOCK_NAME = "2.5 Beta Hot Stock"
HEDGE_NAME = "Hedge"

BOTTOM_DECILE = 0.10

TOP_Y_RANGE = [0, 1000]

FIT_START_DATE = None
FIT_END_DATE = None

USE_RISK_FREE_CAPM_DRIFT = True

# ============================================================
# Hedge strategy parameters
# ============================================================

HEDGE_RESET_DAYS = 63
HEDGE_FLOOR_LOSS = 0.10
HEDGE_UPSIDE_CAP = 0.28
HEDGE_CARRY_ANNUAL = 0.00
HEDGE_REDEPLOY_TRIGGER = -0.05
HEDGE_REDEPLOY_RATE = 1.00

# ============================================================
# Load data
# ============================================================

df = pd.read_csv(CSV_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

required_cols = ["date", "spy_close"]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"CSV is missing required columns: {missing_cols}")

fit_df = df.copy()

if FIT_START_DATE is not None:
    fit_df = fit_df[fit_df["date"] >= FIT_START_DATE]

if FIT_END_DATE is not None:
    fit_df = fit_df[fit_df["date"] <= FIT_END_DATE]

fit_df = fit_df.reset_index(drop=True)

# ============================================================
# GBM parameter estimation
# ============================================================

def fit_gbm_params_from_prices(data, price_col, trading_days_per_year=252):
    asset_df = data[["date", price_col]].dropna().copy()
    asset_df = asset_df.sort_values("date").reset_index(drop=True)

    if len(asset_df) < 3:
        raise ValueError(f"Not enough price observations for {price_col}")

    asset_df["simple_return"] = asset_df[price_col].pct_change()
    asset_df["log_return"] = np.log(asset_df[price_col] / asset_df[price_col].shift(1))

    returns_df = asset_df.dropna(subset=["simple_return", "log_return"]).copy()

    daily_log_mean = returns_df["log_return"].mean()
    daily_log_var = returns_df["log_return"].var(ddof=1)

    annual_log_growth = daily_log_mean * trading_days_per_year
    annual_variance = daily_log_var * trading_days_per_year
    annual_vol = np.sqrt(annual_variance)

    mu_annual = annual_log_growth + 0.5 * annual_variance
    vol_drag = 0.5 * annual_variance

    params = {
        "name": price_col,
        "price_col": price_col,
        "start_date": asset_df["date"].iloc[0],
        "end_date": asset_df["date"].iloc[-1],
        "n_obs": len(returns_df),

        "daily_log_mean": daily_log_mean,
        "daily_log_variance": daily_log_var,

        "mu_annual": mu_annual,
        "annual_log_growth": annual_log_growth,
        "annual_variance": annual_variance,
        "annual_vol": annual_vol,
        "vol_drag": vol_drag,

        "arithmetic_expected_annual_return": np.exp(mu_annual) - 1,
        "geometric_expected_annual_return": np.exp(annual_log_growth) - 1,
    }

    return params, returns_df


spy_params, spy_returns_df = fit_gbm_params_from_prices(
    fit_df,
    "spy_close",
    TRADING_DAYS_PER_YEAR
)

# ============================================================
# Generate 2.5 beta hot stock strategy from SPY parameters
# ============================================================

def make_beta_strategy_params(
    base_params,
    beta=2.5,
    risk_free_rate=0.02,
    use_capm_drift=True,
    name="2.5 Beta Hot Stock"
):
    spy_mu = base_params["mu_annual"]
    spy_sigma = base_params["annual_vol"]

    if use_capm_drift:
        hot_mu = risk_free_rate + beta * (spy_mu - risk_free_rate)
    else:
        hot_mu = beta * spy_mu

    hot_sigma = beta * spy_sigma
    hot_variance = hot_sigma ** 2
    hot_vol_drag = 0.5 * hot_variance
    hot_annual_log_growth = hot_mu - hot_vol_drag

    hot_params = {
        "name": name,
        "beta": beta,
        "start_date": base_params["start_date"],
        "end_date": base_params["end_date"],
        "n_obs": base_params["n_obs"],

        "mu_annual": hot_mu,
        "annual_log_growth": hot_annual_log_growth,
        "annual_variance": hot_variance,
        "annual_vol": hot_sigma,
        "vol_drag": hot_vol_drag,

        "arithmetic_expected_annual_return": np.exp(hot_mu) - 1,
        "geometric_expected_annual_return": np.exp(hot_annual_log_growth) - 1,
    }

    return hot_params


hot_params = make_beta_strategy_params(
    base_params=spy_params,
    beta=HOT_STOCK_BETA,
    risk_free_rate=RISK_FREE_RATE,
    use_capm_drift=USE_RISK_FREE_CAPM_DRIFT,
    name=HOT_STOCK_NAME
)

# ============================================================
# GBM simulation
# ============================================================

def simulate_gbm_paths(
    mu_annual,
    sigma_annual,
    years=10,
    trading_days_per_year=252,
    n_paths=5000,
    s0=100,
    seed=42
):
    rng = np.random.default_rng(seed)

    n_days = years * trading_days_per_year
    dt = 1 / trading_days_per_year

    z = rng.normal(size=(n_days, n_paths))

    log_returns = (
        (mu_annual - 0.5 * sigma_annual**2) * dt
        + sigma_annual * np.sqrt(dt) * z
    )

    log_paths = np.vstack([
        np.zeros(n_paths),
        np.cumsum(log_returns, axis=0)
    ])

    paths = s0 * np.exp(log_paths)

    return paths, log_returns


hot_paths, hot_sim_log_returns = simulate_gbm_paths(
    mu_annual=hot_params["mu_annual"],
    sigma_annual=hot_params["annual_vol"],
    years=SIM_YEARS,
    trading_days_per_year=TRADING_DAYS_PER_YEAR,
    n_paths=N_SIM_PATHS,
    s0=INITIAL_INDEX_VALUE,
    seed=SEED + 1
)

last_data_date = spy_params["end_date"]

sim_dates = pd.bdate_range(
    start=last_data_date + pd.offsets.BDay(1),
    periods=SIM_DAYS + 1
)

t_years = np.arange(SIM_DAYS + 1) / TRADING_DAYS_PER_YEAR

# ============================================================
# Hedge strategy simulation
# ============================================================

def simulate_hedge_strategy_from_underlying(
    underlying_paths,
    reset_days=63,
    floor_loss=0.10,
    upside_cap=0.28,
    carry_annual=0.00,
    redeploy_trigger=-0.05,
    redeploy_rate=1.00,
    trading_days_per_year=252,
    s0=100
):
    """
    Systematic Hedge framework:

    1. Start with exposure to the 2.5-beta underlying.
    2. At every reset date, define a downside floor and an upside give-up level.
    3. During large selloffs, the downside floor creates monetizable cash.
    4. When a reset period ends below the redeploy trigger,
       positive hedge cash is used to buy more shares.
    5. During strong rallies, gains above the upside level are given up.
    """

    n_steps, n_paths = underlying_paths.shape
    strategy_values = np.zeros_like(underlying_paths)

    for path_idx in range(n_paths):
        shares = s0 / underlying_paths[0, path_idx]
        cash = 0.0

        strategy_values[0, path_idx] = s0

        start = 0

        while start < n_steps - 1:
            end = min(start + reset_days, n_steps - 1)

            start_price = underlying_paths[start, path_idx]
            floor_strike = start_price * (1 - floor_loss)
            cap_strike = start_price * (1 + upside_cap)

            period_notional = shares * start_price
            period_length = end - start

            for t in range(start + 1, end + 1):
                current_price = underlying_paths[t, path_idx]

                floor_value = max(floor_strike - current_price, 0) * shares
                cap_value = max(current_price - cap_strike, 0) * shares

                elapsed = t - start
                carry_cost = period_notional * carry_annual * (elapsed / trading_days_per_year)

                marked_overlay_value = floor_value - cap_value - carry_cost

                strategy_values[t, path_idx] = max(
                    0.01,
                    shares * current_price + cash + marked_overlay_value
                )

            end_price = underlying_paths[end, path_idx]
            period_return = end_price / start_price - 1

            floor_payoff = max(floor_strike - end_price, 0) * shares
            cap_payment = max(end_price - cap_strike, 0) * shares
            carry_cost = period_notional * carry_annual * (period_length / trading_days_per_year)

            net_cash_flow = floor_payoff - cap_payment - carry_cost
            cash += net_cash_flow

            if cash > 0 and period_return <= redeploy_trigger:
                dollars_to_redeploy = cash * redeploy_rate
                shares += dollars_to_redeploy / end_price
                cash -= dollars_to_redeploy

            if cash < 0:
                dollars_needed = -cash
                shares_to_sell = min(shares * 0.999, dollars_needed / end_price)

                shares -= shares_to_sell
                cash += shares_to_sell * end_price

                if cash < 0:
                    cash = 0.0

            strategy_values[end, path_idx] = max(
                0.01,
                shares * end_price + cash
            )

            start = end

    return strategy_values


hedge_paths = simulate_hedge_strategy_from_underlying(
    underlying_paths=hot_paths,
    reset_days=HEDGE_RESET_DAYS,
    floor_loss=HEDGE_FLOOR_LOSS,
    upside_cap=HEDGE_UPSIDE_CAP,
    carry_annual=HEDGE_CARRY_ANNUAL,
    redeploy_trigger=HEDGE_REDEPLOY_TRIGGER,
    redeploy_rate=HEDGE_REDEPLOY_RATE,
    trading_days_per_year=TRADING_DAYS_PER_YEAR,
    s0=INITIAL_INDEX_VALUE
)

# ============================================================
# Bottom-decile cohort selection
# ============================================================

def bottom_decile_indices(paths, bottom_decile=0.10):
    final_values = paths[-1, :]
    cutoff = np.percentile(final_values, bottom_decile * 100)
    return np.where(final_values <= cutoff)[0], cutoff


hedge_bottom_indices, hedge_bottom_cutoff = bottom_decile_indices(
    hedge_paths,
    BOTTOM_DECILE
)

hot_bottom_indices, hot_bottom_cutoff = bottom_decile_indices(
    hot_paths,
    BOTTOM_DECILE
)

display_rng = np.random.default_rng(SEED + 999)

hedge_display_indices = display_rng.choice(
    hedge_bottom_indices,
    size=min(N_DISPLAY_PATHS, len(hedge_bottom_indices)),
    replace=False
)

hot_display_indices = display_rng.choice(
    hot_bottom_indices,
    size=min(N_DISPLAY_PATHS, len(hot_bottom_indices)),
    replace=False
)

# ============================================================
# Conditional arithmetic / geometric growth trends
# ============================================================

def conditional_ag_trends(paths, selected_indices, s0=100):
    """
    Conditional A/G trends for a cohort of paths.

    Arithmetic trend:
        Cross-sectional arithmetic mean value through time.

    Geometric trend:
        Cross-sectional geometric mean value through time.

    The visual gap between the two is the volatility-drag story:
        arithmetic outcomes can look acceptable while geometric wealth collapses.
    """

    cohort = paths[:, selected_indices]

    arithmetic_trend = np.nanmean(cohort, axis=1)

    safe_cohort = np.maximum(cohort, 1e-12)
    geometric_trend = s0 * np.exp(
        np.nanmean(np.log(safe_cohort / s0), axis=1)
    )

    return arithmetic_trend, geometric_trend


hedge_cond_arith_path, hedge_cond_geo_path = conditional_ag_trends(
    hedge_paths,
    hedge_bottom_indices,
    INITIAL_INDEX_VALUE
)

hot_cond_arith_path, hot_cond_geo_path = conditional_ag_trends(
    hot_paths,
    hot_bottom_indices,
    INITIAL_INDEX_VALUE
)

# ============================================================
# Conditional summaries and stats
# ============================================================

def summarize_selected_paths(paths, selected_indices):
    cohort = paths[:, selected_indices]

    return {
        "p10": np.percentile(cohort, 10, axis=1),
        "p50": np.percentile(cohort, 50, axis=1),
        "p90": np.percentile(cohort, 90, axis=1),
        "mean_mc": cohort.mean(axis=1)
    }


hedge_bottom_summary = summarize_selected_paths(
    hedge_paths,
    hedge_bottom_indices
)

hot_bottom_summary = summarize_selected_paths(
    hot_paths,
    hot_bottom_indices
)


def calc_path_mdds(paths):
    running_max = np.maximum.accumulate(paths, axis=0)
    drawdowns = paths / running_max - 1
    max_drawdowns = drawdowns.min(axis=0)

    return -max_drawdowns


def calc_realized_vol_drag(paths, selected_indices, trading_days_per_year=252):
    cohort = paths[:, selected_indices]

    daily_returns = cohort[1:, :] / cohort[:-1, :] - 1
    daily_returns = np.where(np.isfinite(daily_returns), daily_returns, np.nan)
    daily_returns = np.clip(daily_returns, -0.999999, None)

    path_arith_growth = np.nanmean(daily_returns, axis=0) * trading_days_per_year
    path_geo_growth = np.nanmean(np.log1p(daily_returns), axis=0) * trading_days_per_year

    path_vol_drag = path_arith_growth - path_geo_growth

    return np.nanmean(path_vol_drag)


def calc_conditioned_stats(paths, selected_indices, years, s0=100):
    cohort = paths[:, selected_indices]
    final_values = cohort[-1, :]

    path_cagrs = (final_values / s0) ** (1 / years) - 1
    path_mdds = calc_path_mdds(cohort)

    stats = {
        "avg_cagr": np.nanmean(path_cagrs),
        "avg_mdd": np.nanmean(path_mdds),
        "vol_drag": calc_realized_vol_drag(paths, selected_indices, TRADING_DAYS_PER_YEAR),
        "terminal_1pct_value": np.percentile(final_values, 1),
        "mean_final": np.nanmean(final_values),
        "median_final": np.nanmedian(final_values),
        "p10_final": np.percentile(final_values, 10),
        "p90_final": np.percentile(final_values, 90),
        "bottom_cutoff": np.percentile(paths[-1, :], BOTTOM_DECILE * 100),
    }

    return stats


hedge_bottom_stats = calc_conditioned_stats(
    paths=hedge_paths,
    selected_indices=hedge_bottom_indices,
    years=SIM_YEARS,
    s0=INITIAL_INDEX_VALUE
)

hot_bottom_stats = calc_conditioned_stats(
    paths=hot_paths,
    selected_indices=hot_bottom_indices,
    years=SIM_YEARS,
    s0=INITIAL_INDEX_VALUE
)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

principal_color = "#777777"

below_principal_color = "#ff4444"
above_principal_color = "#ffd84d"

arith_color = "#33aaff"
geo_color = "#ff66cc"

band_color_hedge = "rgba(0,255,136,0.12)"
band_color_hot = "rgba(255,170,51,0.12)"

percent_bar_color = "#33aaff"
value_bar_color = "#ff6666"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# ============================================================
# Bar chart data
# ============================================================

percent_stat_labels = [
    "Bottom 10%<br>Avg. CAGR",
    "Bottom 10%<br>Avg. MDD",
    "Bottom 10%<br>Vol Drag"
]

value_stat_labels = [
    "Bottom 10%<br>1% Value"
]

def percent_values_from_stats(stats):
    return [
        stats["avg_cagr"] * 100,
        stats["avg_mdd"] * 100,
        stats["vol_drag"] * 100
    ]

def value_values_from_stats(stats):
    return [
        stats["terminal_1pct_value"]
    ]

def percent_text(values):
    return [f"{v:.1f}%" for v in values]

def value_text(values):
    return [f"{v:.0f}" for v in values]

hedge_percent_values = percent_values_from_stats(hedge_bottom_stats)
hot_percent_values = percent_values_from_stats(hot_bottom_stats)

hedge_value_values = value_values_from_stats(hedge_bottom_stats)
hot_value_values = value_values_from_stats(hot_bottom_stats)

all_percent_values = hedge_percent_values + hot_percent_values
all_value_values = hedge_value_values + hot_value_values

percent_axis_min = min(0, min(all_percent_values) * 1.25)
percent_axis_max = max(10, max(all_percent_values) * 1.25)

value_axis_min = 0
value_axis_max = max(100, max(all_value_values) * 1.35)

# ============================================================
# Plot helpers
# ============================================================

def split_path_by_principal(path, principal=100):
    below = np.where(path < principal, path, np.nan)
    above = np.where(path >= principal, path, np.nan)

    return below, above


def make_bottom_decile_title(name, stats):
    return (
        f"<b>{name}: Bottom 10% Terminal Paths</b><br>"
        f"Avg CAGR: {stats['avg_cagr']:.1%} · "
        f"Avg MDD: {stats['avg_mdd']:.1%} · "
        f"Vol Drag: {stats['vol_drag']:.1%}<br>"
        f"Bottom-decile cutoff: {stats['bottom_cutoff']:.0f} · "
        f"1% Value inside cohort: {stats['terminal_1pct_value']:.0f}"
    )


hedge_subtitle = make_bottom_decile_title(
    HEDGE_NAME,
    hedge_bottom_stats
)

hot_subtitle = make_bottom_decile_title(
    HOT_STOCK_NAME,
    hot_bottom_stats
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.50, 0.50],
    row_heights=[0.68, 0.32],
    horizontal_spacing=0.10,
    vertical_spacing=0.18,
    specs=[
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"secondary_y": True}, {"secondary_y": True}]
    ],
    subplot_titles=(
        hedge_subtitle,
        hot_subtitle,
        f"{HEDGE_NAME}: Bottom 10% Metrics",
        f"{HOT_STOCK_NAME}: Bottom 10% Metrics"
    )
)

animated_y_series = []

def add_animated_trace(row, col, y_series, trace):
    fig.add_trace(trace, row=row, col=col)
    animated_y_series.append(y_series)

# ============================================================
# Top row: Hedge bottom-decile paths
# ============================================================

fig.add_hline(
    y=INITIAL_INDEX_VALUE,
    line=dict(color=principal_color, width=1, dash="dash"),
    opacity=0.80,
    row=1,
    col=1
)

add_animated_trace(
    1,
    1,
    hedge_bottom_summary["p90"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_bottom_summary["p90"][0]],
        mode="lines",
        line=dict(color="rgba(0,255,136,0.0)", width=0),
        showlegend=False,
        hoverinfo="skip",
        name=f"{HEDGE_NAME} Bottom P90"
    )
)

add_animated_trace(
    1,
    1,
    hedge_bottom_summary["p10"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_bottom_summary["p10"][0]],
        mode="lines",
        line=dict(color="rgba(0,255,136,0.0)", width=0),
        fill="tonexty",
        fillcolor=band_color_hedge,
        showlegend=False,
        hoverinfo="skip",
        name=f"{HEDGE_NAME} Bottom 10% Band"
    )
)

for path_num, path_idx in enumerate(hedge_display_indices, start=1):
    y_path = hedge_paths[:, path_idx]
    y_below, y_above = split_path_by_principal(y_path, INITIAL_INDEX_VALUE)

    add_animated_trace(
        1,
        1,
        y_below,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_below[0]],
            mode="lines",
            line=dict(color=below_principal_color, width=1.5),
            opacity=0.90,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HEDGE_NAME} Below Principal Path {path_num}"
        )
    )

    add_animated_trace(
        1,
        1,
        y_above,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_above[0]],
            mode="lines",
            line=dict(color=above_principal_color, width=1.5),
            opacity=0.90,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HEDGE_NAME} Above Principal Path {path_num}"
        )
    )

add_animated_trace(
    1,
    1,
    hedge_cond_arith_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_cond_arith_path[0]],
        mode="lines",
        line=dict(color=arith_color, width=4),
        showlegend=False,
        name=f"{HEDGE_NAME} Bottom-Decile Arithmetic Trend",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Bottom 10% arithmetic trend: %{y:.2f}<extra></extra>"
        )
    )
)

add_animated_trace(
    1,
    1,
    hedge_cond_geo_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_cond_geo_path[0]],
        mode="lines",
        line=dict(color=geo_color, width=4, dash="dash"),
        showlegend=False,
        name=f"{HEDGE_NAME} Bottom-Decile Geometric Trend",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Bottom 10% geometric trend: %{y:.2f}<extra></extra>"
        )
    )
)

# ============================================================
# Top row: Hot stock bottom-decile paths
# ============================================================

fig.add_hline(
    y=INITIAL_INDEX_VALUE,
    line=dict(color=principal_color, width=1, dash="dash"),
    opacity=0.80,
    row=1,
    col=2
)

add_animated_trace(
    1,
    2,
    hot_bottom_summary["p90"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_bottom_summary["p90"][0]],
        mode="lines",
        line=dict(color="rgba(255,170,51,0.0)", width=0),
        showlegend=False,
        hoverinfo="skip",
        name=f"{HOT_STOCK_NAME} Bottom P90"
    )
)

add_animated_trace(
    1,
    2,
    hot_bottom_summary["p10"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_bottom_summary["p10"][0]],
        mode="lines",
        line=dict(color="rgba(255,170,51,0.0)", width=0),
        fill="tonexty",
        fillcolor=band_color_hot,
        showlegend=False,
        hoverinfo="skip",
        name=f"{HOT_STOCK_NAME} Bottom 10% Band"
    )
)

for path_num, path_idx in enumerate(hot_display_indices, start=1):
    y_path = hot_paths[:, path_idx]
    y_below, y_above = split_path_by_principal(y_path, INITIAL_INDEX_VALUE)

    add_animated_trace(
        1,
        2,
        y_below,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_below[0]],
            mode="lines",
            line=dict(color=below_principal_color, width=1.5),
            opacity=0.90,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HOT_STOCK_NAME} Below Principal Path {path_num}"
        )
    )

    add_animated_trace(
        1,
        2,
        y_above,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_above[0]],
            mode="lines",
            line=dict(color=above_principal_color, width=1.5),
            opacity=0.90,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HOT_STOCK_NAME} Above Principal Path {path_num}"
        )
    )

add_animated_trace(
    1,
    2,
    hot_cond_arith_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_cond_arith_path[0]],
        mode="lines",
        line=dict(color=arith_color, width=4),
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Bottom-Decile Arithmetic Trend",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Bottom 10% arithmetic trend: %{y:.2f}<extra></extra>"
        )
    )
)

add_animated_trace(
    1,
    2,
    hot_cond_geo_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_cond_geo_path[0]],
        mode="lines",
        line=dict(color=geo_color, width=4, dash="dash"),
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Bottom-Decile Geometric Trend",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Bottom 10% geometric trend: %{y:.2f}<extra></extra>"
        )
    )
)

# ============================================================
# Bottom row: conditioned metric bar charts
# ============================================================

fig.add_trace(
    go.Bar(
        x=percent_stat_labels,
        y=hedge_percent_values,
        marker=dict(color=percent_bar_color, opacity=0.85),
        text=percent_text(hedge_percent_values),
        textposition="outside",
        showlegend=False,
        name=f"{HEDGE_NAME} Bottom 10% Percent Metrics",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=2,
    col=1,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=value_stat_labels,
        y=hedge_value_values,
        marker=dict(color=value_bar_color, opacity=0.85),
        text=value_text(hedge_value_values),
        textposition="outside",
        showlegend=False,
        name=f"{HEDGE_NAME} Bottom 10% 1% Value",
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>"
    ),
    row=2,
    col=1,
    secondary_y=True
)

fig.add_trace(
    go.Bar(
        x=percent_stat_labels,
        y=hot_percent_values,
        marker=dict(color=percent_bar_color, opacity=0.85),
        text=percent_text(hot_percent_values),
        textposition="outside",
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Bottom 10% Percent Metrics",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=2,
    col=2,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=value_stat_labels,
        y=hot_value_values,
        marker=dict(color=value_bar_color, opacity=0.85),
        text=value_text(hot_value_values),
        textposition="outside",
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Bottom 10% 1% Value",
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>"
    ),
    row=2,
    col=2,
    secondary_y=True
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

n_top_traces = len(animated_y_series)

frame_indices = list(range(1, len(sim_dates), FRAME_STRIDE))

if frame_indices[-1] != len(sim_dates) - 1:
    frame_indices.append(len(sim_dates) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    x_slice = sim_dates[:i + 1]

    frame_data = []

    for y_series in animated_y_series:
        frame_data.append(
            go.Scatter(
                x=x_slice,
                y=y_series[:i + 1]
            )
        )

    frame_data.extend([
        go.Bar(
            x=percent_stat_labels,
            y=hedge_percent_values,
            text=percent_text(hedge_percent_values),
            textposition="outside",
            marker=dict(color=percent_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=value_stat_labels,
            y=hedge_value_values,
            text=value_text(hedge_value_values),
            textposition="outside",
            marker=dict(color=value_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=percent_stat_labels,
            y=hot_percent_values,
            text=percent_text(hot_percent_values),
            textposition="outside",
            marker=dict(color=percent_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=value_stat_labels,
            y=hot_value_values,
            text=value_text(hot_value_values),
            textposition="outside",
            marker=dict(color=value_bar_color, opacity=0.85),
            showlegend=False
        )
    ])

    frames.append(
        go.Frame(
            data=frame_data,
            traces=list(range(n_top_traces + 4)),
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": ANIMATION_REDRAW},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": sim_dates[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="Bottom 10% Paths: Volatility Drag, Arithmetic Growth, and Geometric Wealth Destruction",
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=950,
    width=1200,
    margin=dict(t=170, b=150, r=80, l=75),
    showlegend=False,
    hovermode="closest",
    barmode="group",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": ANIMATION_REDRAW},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": ANIMATION_REDRAW},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=12))

# ============================================================
# Axes: top row
# ============================================================

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[sim_dates[0], sim_dates[-1]],
    title_text="Simulation Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=TOP_Y_RANGE,
    title_text="Bottom 10% Simulated Value, Start = 100"
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[sim_dates[0], sim_dates[-1]],
    title_text="Simulation Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=TOP_Y_RANGE,
    title_text="Bottom 10% Simulated Value, Start = 100"
)

# ============================================================
# Axes: bottom row
# ============================================================

fig.update_xaxes(
    axis_style,
    row=2,
    col=1,
    tickfont=dict(color=off_white, size=10),
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    secondary_y=False,
    range=[percent_axis_min, percent_axis_max],
    title_text="Bottom 10% Percent Metrics",
    ticksuffix="%"
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    secondary_y=True,
    range=[value_axis_min, value_axis_max],
    title_text="",
    showgrid=False
)

fig.update_xaxes(
    axis_style,
    row=2,
    col=2,
    tickfont=dict(color=off_white, size=10),
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    secondary_y=False,
    range=[percent_axis_min, percent_axis_max],
    title_text="Bottom 10% Percent Metrics",
    ticksuffix="%"
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    secondary_y=True,
    range=[value_axis_min, value_axis_max],
    title_text="",
    showgrid=False
)

fig.update_traces(
    cliponaxis=False,
    selector=dict(type="bar")
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("bottom_decile_vol_drag_hedge_vs_hot.html", include_plotlyjs="cdn")

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
 - This notebook treats stock returns as random variables whose distribution is a *compression of information* that changes over time, and shows why the tidy return histogram taught in classrooms hides the extreme, market-moving events—Black Swans—that actually determine survival.
 - It calibrates a static Normal distribution (mean $\mu$, standard deviation $\sigma$) to SPY returns and models tail events as a geometric random variable, demonstrating that this parametric approach severely *underestimates* tail risk due to return autocorrelation, excess kurtosis, and other violations of its assumptions.
 - Layering on complexity does not cure model and parameter risk: conditional (regime) models such as $\text{GARCH}(1,1)$—$\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2$—capture time-varying volatility more realistically, yet still cannot reliably anticipate the next extreme move.
 - Empirical, walk-forward probabilities fail for the same reason: no amount of statistics or backtesting can predict the future, because the drivers of extreme returns (news, sentiment, geopolitics, macro, regulation) are absent from the price series itself.
 - The key message: the purpose of modeling is **not** prediction but *positioning and survival*—like a casino, we build effective statistics to size and hedge exposure so that no single state of the world can wipe us out.

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$